# Fase 1: Dependencias

In [ ]:
# Instalamos los paquetes recomendados con Conda según lo indicado en:
# https://github.com/HathewayWill/WRF_Python_Scripts

import subprocess
result = subprocess.run([
    'conda', 'install', '-c', 'conda-forge', '-y',
    'numpy', 'matplotlib', 'metpy', 'wrf-python',
    'netCDF4', 'pillow', 'scipy', 'cartopy', 'geopandas',
    'seaborn', 'pandas', 'xarray', 'tqdm'
], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [ ]:
# 1. Forzamos a Linux a cargar el libgomp completo de tu entorno Conda
import ctypes
ctypes.CDLL("/home/gcamilo/envs/wrf312/lib/libgomp.so.1", mode=ctypes.RTLD_GLOBAL)

# 2. Ahora sí, importamos wrf
import wrf
print("WRF versión:", wrf.__version__)

# 3. Y luego podemos importar PyTorch sin que rompa nada
import torch
print("PyTorch versión:", torch.__version__)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as crs
import cartopy.feature as cfeature
import os
import xarray as xr
import re
import torch
import pandas as pd
import seaborn as sns
import gc
import shutil
from tqdm.notebook import tqdm                 # Para mostrar progreso
from netCDF4 import Dataset
from wrf import getvar, to_np, get_cartopy, latlon_coords, xy_to_ll, ll_to_xy, geo_bounds
from copy import deepcopy
from metpy import constants
from collections import defaultdict
from datetime import datetime, timedelta
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

# Fase 2: Análisis de datos

In [ ]:
ruta_viejos = os.path.expanduser("~/jupyter/salidasWRF2018/wrfout*")
all_files = sorted(glob.glob(ruta_viejos))
print(all_files[0])
ds = Dataset(all_files[0])
print(ds.__dict__)

In [ ]:
# Listamos todas las variables disponibles
for var in sorted(ds.variables.keys()):
    print(var)

In [ ]:
# Vemos la resolución espacial
print(ds.DX, ds.DY)

In [ ]:
# Vemos el tipo de la proyección
print(ds.MAP_PROJ_CHAR)  # Nombre de la proyección
print(ds.MAP_PROJ)       # Código numérico

In [ ]:
temperature = getvar(ds, "T")

# Print con metadata 
print(temperature) # por default, meta = True 

print('\n' + '='*100 + '\n')

# Print sin metadata
temperature_no_meta = getvar(ds, "T", meta=False)
print (temperature_no_meta)

In [ ]:
# Transformar xarrays a arrays de numpy

# Si la variable no tiene valores faltantes o inválidos, el resultado es un array de datos
temperature_arr = to_np(temperature)
print(type(temperature_arr))

# En caso contrario, es resultado es un array de datos más una máscara (conocido como MaskedArray)
cape_3d = getvar(ds, "cape_3d")
cape_3d_masked_array = to_np(cape_3d)
print(type(cape_3d_masked_array))
print(cape_3d_masked_array)       # Parte con los datos
print(cape_3d_masked_array.mask)  # Parte con la máscara (True = inválido, False = válido)

# Podemos rellenar los valores faltantes con un valor arbitrario para tener simplemente un array de datos
cape_3d_array = np.ma.filled(cape_3d_masked_array, fill_value=45.0)
print(type(cape_3d_array))
print(cape_3d_array)

In [ ]:
# Pasar de puntos de grilla a latitudes y viceversa

# Una sola coordenada
lat_lon = xy_to_ll(ds, 400, 200)
print(lat_lon, '\n')

x_y = ll_to_xy(ds, lat_lon[0], lat_lon[1])
print (x_y, '\n')

# Múltiples coordenadas
lat_lon = xy_to_ll(ds, [400,105], [200,205])
print(lat_lon, '\n')

x_y = ll_to_xy(ds, lat_lon[0,:], lat_lon[1,:])
print (x_y)

In [ ]:
# Plotear una variable

# 1. Elegir la variable a plotear (usaremos slp como ejemplo)
slp = getvar(ds, "slp")

# 2. Obtener el objeto cartopy mapping
cart_proj = get_cartopy(slp)
print(type(cart_proj))

# 3. Obtener latitud y longitud
lats, lons = latlon_coords(slp)

# 4. Plotear
# 4.1. Crear figura con proyección
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=cart_proj)
# 4.2. Agregar características geográficas
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.STATES)
# 4.3. Graficar la variable
plt.contourf(to_np(lons), to_np(lats), to_np(slp),
             transform=crs.PlateCarree(),  # para que cartopy sepa que las coordenadas estan en lat/lon
             cmap="jet", levels=20)
plt.colorbar(label="SLP (hPa)")
plt.title("Sea Level Pressure")
plt.savefig("slp.png", dpi=150)
plt.show()

In [ ]:
# Animación de una variable a lo largo de un día

# 1. Cargar todos los archivos de un día (ejemplo: 2018-11-10)
files_18_11_10 = sorted(glob.glob(os.path.expanduser("~/jupyter/salidasWRF2018/wrfout_d01_2018-11-10*")))
print(f"Archivos encontrados: {len(files_18_11_10)}")

# 2. Calcular min/max globales para que la escala de colores sea consistente
slp_min, slp_max = float("inf"), float("-inf")
for f in tqdm(files_18_11_10, desc="Calculando min/max"):
    ds = Dataset(f)
    slp = to_np(getvar(ds, "slp"))
    slp_min = min(slp_min, slp.min())
    slp_max = max(slp_max, slp.max())
    ds.close()
print(f"Rango SLP: {slp_min:.1f} - {slp_max:.1f} hPa")

# 3. Preparar figura
ds0 = Dataset(files_18_11_10[0])       # todos los archivos tienen la misma proyección cartográfica y las mismas coordenadas lat/lon
slp0 = getvar(ds0, "slp")
cart_proj = get_cartopy(slp0)
lats, lons = latlon_coords(slp0)
ds0.close()

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=cart_proj)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.STATES)

levels = 20                            # valor arbitraio, representa el número de colores en el mapa

# 4. Función de actualización para cada frame
pbar = tqdm(total=len(files_18_11_10), desc="Generando frames")

def update(frame_idx):
    pbar.update(1)
    
    ax.clear()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.COASTLINE)
    ax.add_feature(cfeature.STATES)

    ds = Dataset(files_18_11_10[frame_idx])
    slp = getvar(ds, "slp")
    tiempo = str(slp.Time.values)[:16]       # Solo muestra hasta los minutos
    ds.close()

    cf = ax.contourf(to_np(lons), to_np(lats), to_np(slp),
                     transform=crs.PlateCarree(),
                     cmap="jet", levels=levels,
                     vmin=slp_min, vmax=slp_max)

    ax.set_title(f"Sea Level Pressure — {tiempo} UTC")
    return cf

# 5. Crear animación
ani = animation.FuncAnimation(fig, update,
                               frames=len(files_18_11_10),
                               interval=200)  # 200ms entre frames

# Agregar colorbar (una sola vez)
cf_inicial = update(0)
plt.colorbar(cf_inicial, ax=ax, label="SLP (hPa)")

# 6. Guardar como GIF o MP4
ani.save("slp_animacion.gif", writer="pillow", fps=5)
# ani.save("slp_animacion.mp4", writer="ffmpeg", fps=5)

plt.show()

In [ ]:
# También podemos limitar la parte del mapa que queremos graficar

# Coordenadas aproximadas de la provincia de Córdoba
LAT_MIN = -34.00   # Sur
LAT_MAX = -29.42   # Norte  
LON_MIN = -65.73   # Oeste
LON_MAX = -61.79   # Este

# Figura igual que antes
fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=cart_proj)
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.STATES)

# Limitar la vista del mapa con set_extent
ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
              crs=crs.PlateCarree())

plt.contourf(to_np(lons), to_np(lats), to_np(slp),
             transform=crs.PlateCarree(),
             cmap="jet", levels=20)
plt.colorbar(label="SLP (hPa)")
plt.title("Sea Level Pressure - Provincia de Córdoba")
plt.savefig("slp_cordoba.png", dpi=150)
plt.show()


# Fase 3: Filtrado de datos

| Variable | Descripción | Clave | Unidades | Obtención |
|---|---|---|---|---|
| ~~REFL_10CM~~ | ~~Reflectividad radar simulada a 10 cm de longitud de onda~~ | ~~`"REFL_10CM"`~~ | ~~dBZ~~ | ~~Directa del wrfout~~ |
| MDBZ | Reflectividad radar máxima en la columna | `"mdbz"` | dBZ | Calculada por wrf-python |
| LIGHTDENS | Densidad de descargas eléctricas (variable objetivo) | `"LIGHTDENS"` | flash/columna | Directa del wrfout |
| QICE | Razón de mezcla de cristales de hielo | `"QICE"` | kg/kg | Directa del wrfout (se filtran en el rango [-40, -10]C°) |
| QHAIL | Razón de mezcla de granizo | `"QHAIL"` | kg/kg | Directa del wrfout (se filtran en el rango [-40, -10]C°) |
| QGRAUP | Razón de mezcla de graupel | `"QGRAUP"` | kg/kg | Directa del wrfout (se filtran en el rango [-40, -10]C°) |
| QSNOW | Razón de mezcla de nieve | `"QSNOW"` | kg/kg | Directa del wrfout (se filtran en el rango [-40, -10]C°) |
| ~~QCLOUD~~ | ~~Razón de mezcla de agua líquida en nube~~ | ~~`"QCLOUD"`~~ | ~~kg/kg~~ | ~~Directa del wrfout~~ |
| ~~QRAIN~~ | ~~Razón de mezcla de lluvia~~ | ~~`"QRAIN"`~~ | ~~kg/kg~~ | ~~Directa del wrfout~~ |
| CTT | Temperatura del tope de nube | `"ctt"` | °C | Calculada por wrf-python |
| ~~LPI~~ | ~~Índice de potencial de rayos~~ | ~~—~~ | ~~J/kg~~ | ~~No disponible en wrfout, requiere cálculo manual~~ |
| ~~WMAX~~ | ~~Máxima velocidad vertical del viento en la columna~~ | ~~`"wa"`~~ | ~~m/s~~ | ~~Calculada por wrf-python~~ |

In [ ]:
files_18_11_10 = sorted(glob.glob(os.path.expanduser("~/jupyter/salidasWRF2018/wrfout_d01_2018-11-10*")))

In [ ]:
# Parche para poder usar los parámetros use_varint y use_liqskin para mdbz en get_var

from wrf.routines import _VALID_KARGS
_VALID_KARGS["maxdbz"] = ["use_varint", "use_liqskin"]

In [ ]:
def transform_Q(ds_Q, var):
    for v in var:
        ds_Q[v] = ds_Q[v].where(ds_Q[v] >= 0, 0) * 1e-3

    return ds_Q

In [ ]:
def get_LPI(ds):

    ds_LPI = deepcopy(ds[['W', 'T', 'GEOPT', 'QGRAUP', 'QICE', 'QSNOW', 'QRAIN', 'QCLOUD']])

    ds_LPI = transform_Q(ds_LPI, ['QGRAUP', 'QICE', 'QSNOW', 'QRAIN', 'QCLOUD'])

    ds_LPI['GEOPT'] = ds_LPI['GEOPT']/constants.g

    del_Z = ds_LPI['GEOPT'][1:, :, :].values - ds_LPI['GEOPT'][:-1, :, :].values

    ds_LPI = ds_LPI.isel(bottom_top = slice(0, -1))

    flag_T = (ds_LPI['T'] > (-20 + 273.15)) * (ds_LPI['T'] < (273.15))

    ds_LPI = ds_LPI.where(flag_T)

    Q_l = ds_LPI['QCLOUD'] + ds_LPI['QRAIN']

    den_i = ds_LPI['QICE'] + ds_LPI['QGRAUP']
    num_i = np.sqrt(ds_LPI['QICE'] * ds_LPI['QGRAUP'])

    flag_i = (ds_LPI['QICE'] != 0) + (ds_LPI['QGRAUP'] != 0)
    den_i = den_i.where(flag_i, 10000)
    num_i = num_i.where(flag_i,  0)
    
    den_s = ds_LPI['QSNOW'] + ds_LPI['QGRAUP']
    num_s = np.sqrt(ds_LPI['QSNOW'] * ds_LPI['QGRAUP'])

    flag_s = (ds_LPI['QSNOW'] != 0) + (ds_LPI['QGRAUP'] != 0)
    den_s = den_s.where(flag_s, 10000)
    num_s = num_s.where(flag_s,  0)

    q_isg = ds_LPI['QGRAUP']*(num_i/den_i + num_s/den_s)

    num_l = np.sqrt(Q_l * q_isg)
    den_l = Q_l + q_isg

    flag_l = (Q_l != 0) + (q_isg != 0)
    den_l = den_l.where(flag_l, 10000)
    num_l = num_l.where(flag_l,  0)

    lpi_tmp = ((del_Z * (2 * (num_l/den_l)) * ds_LPI['W']**2)).sum(dim = 'bottom_top')

    lpi = lpi_tmp/(del_Z*flag_T).sum(dim = 'bottom_top')

    return lpi

In [ ]:
def calcular_lpi(ds):
    """Adaptador: recibe un Dataset de netCDF4, devuelve LPI como numpy array."""
    variables = ['tk', 'wa', 'geopt', 'QGRAUP', 'QICE', 'QSNOW', 'QRAIN', 'QCLOUD']
    ds_list = []
    for var in variables:
        ds_list.append(getvar(ds, var))
    ds_xr = xr.combine_by_coords(ds_list, compat='override')
    ds_xr = ds_xr.rename({'temp': 'T', 'wa': 'W', 'geopt': 'GEOPT'})
    lpi = get_LPI(ds_xr)
    return to_np(lpi)

def calcular_intice_tmp(ds):
    variables = ['temp', 'QGRAUP', 'QHAIL', 'QICE', 'QSNOW']
    ds_list = [getvar(ds, v, units="degC") if v == 'temp' else getvar(ds, v) for v in variables]
    ds_xr = xr.combine_by_coords(ds_list, compat='override')
    
    qsum = ds_xr['QGRAUP'] + ds_xr['QHAIL'] + ds_xr['QICE'] + ds_xr['QSNOW']
    qsum = qsum.fillna(0)
    
    mask = (ds_xr['temp'] <= -10) & (ds_xr['temp'] >= -40)
    qmask = qsum.where(mask, 0)
    
    return np.array(qmask.sum(dim="bottom_top", skipna=True))

In [ ]:

def animar_variable(archivos, var_config, output_file):
    """
    Genera una animación para una variable, suma de variables o una
    variable calculada mediante una función personalizada.
    
    archivos : list
        Lista de rutas a los archivos wrfout a animar.
    var_config : dict
        Diccionario con la configuración de la variable a extraer:
        - "nombre" (str): Nombre para el título y el colorbar.
        - "unidades" (str): Cadena con las unidades.
        - "cmap" (str): Mapa de colores de matplotlib a usar.
        - "keys" (list, opcional): Lista de claves para getvar. Si hay más de una, 
          se suman sus valores.
        - "func" (callable, opcional): Función personalizada que recibe un dataset 
          y devuelve un array 2D. Si se provee "func", se ignora "keys".
        - "clip_min" (float, opcional): Límite inferior de los datos. Cualquier valor 
          menor a este será reemplazado por este número.
    output_file : str
        Nombre del archivo de salida.
    """
    
    # 1. Calcular min/max globales
    vmin, vmax = float("inf"), float("-inf")
    for f in tqdm(archivos, desc=f"Calculando rango {var_config['nombre']}"):
        with Dataset(f) as ds:
            datos = _extraer(ds, var_config)
            vmin = min(vmin, datos.min())
            vmax = max(vmax, datos.max())
    print(f"Rango {var_config['nombre']}: {vmin:.4f} - {vmax:.4f} {var_config['unidades']}")

    # 2. Preparar figura
    with Dataset(archivos[0]) as ds0:
        ref = getvar(ds0, "LIGHTDENS")  # solo para obtener metadatos (proyección y coordenadas)
        cart_proj = get_cartopy(ref)
        lats, lons = latlon_coords(ref)

    fig = plt.figure(figsize=(10, 8))
    ax = plt.axes(projection=cart_proj)

    # 3. Función de actualización
    pbar = tqdm(total=len(archivos), desc="Generando frames")

    def update(frame_idx):
        ax.clear()
        ax.add_feature(cfeature.BORDERS)
        ax.add_feature(cfeature.COASTLINE)
        ax.add_feature(cfeature.STATES)

        ds = Dataset(archivos[frame_idx])
        datos = _extraer(ds, var_config)
        tiempo = str(getvar(ds, "LIGHTDENS").Time.values)[:16]
        ds.close()

        cf = ax.contourf(to_np(lons), to_np(lats), datos,
                         transform=crs.PlateCarree(),
                         cmap=var_config["cmap"], levels=np.linspace(vmin, vmax, 21),
                         vmin=vmin, vmax=vmax)
        ax.set_title(f"{var_config['nombre']} — {tiempo} UTC")
        pbar.update(1)
        return cf

    # 4. Colorbar y animación
    cf_inicial = update(0)
    plt.colorbar(cf_inicial, ax=ax, label=f"{var_config['nombre']} ({var_config['unidades']})")

    ani = animation.FuncAnimation(fig, update,
                                   frames=range(1, len(archivos)),
                                   interval=200)
    carpeta_destino = "../videos&gifs"
    os.makedirs(carpeta_destino, exist_ok=True) # Crea la carpeta si no existe
    ruta_final = os.path.join(carpeta_destino, output_file)
    ani.save(ruta_final, writer="pillow", fps=5)
    pbar.close()
    print(f"Animación guardada en {ruta_final}")
    plt.show()

def _extraer(ds, var_config):
    """Extrae y suma las variables indicadas en var_config."""
    if "func" in var_config:
        return var_config["func"](ds)
        
    resultado = None
    parametros_extra = var_config.get("getvar_kwargs", {})
    for key in var_config["keys"]:
        datos = to_np(getvar(ds, key, **parametros_extra))
        
        # Si es 3D, tomamos el máximo en el eje vertical
        if datos.ndim == 3:
            datos = datos.max(axis=0)
            
        resultado = datos if resultado is None else resultado + datos
        
    # Clipear valores mínimos si se especifica
    if "clip_min" in var_config:
        resultado = np.maximum(resultado, var_config["clip_min"])
    
    return resultado

In [ ]:
# REFL_10CM
animar_variable(files_18_11_10,
    {"nombre": "Reflectividad", "unidades": "dBZ",
     "cmap": "jet", "keys": ["REFL_10CM"]},
    "refl_animacion.gif")

In [ ]:
# MDBZ
animar_variable(files_18_11_10,
    {"nombre": "Reflectividad Máxima", "unidades": "dBZ",
     "cmap": "jet", "keys": ["mdbz"], "clip_min": 0,
    "getvar_kwargs": {"use_varint": True, "use_liqskin": True}},
    "mdbz_animacion.gif")


In [ ]:
# LIGHTDENS (target)
animar_variable(files_18_11_10,
    {"nombre": "LIGHTDENS", "unidades": "flash/col",
     "cmap": "hot_r", "keys": ["LIGHTDENS"]},
    "lightdens_animacion.gif")

In [ ]:
# QICE + QHAIL + QGRAUP + QSNOW
animar_variable(files_18_11_10,
    {"nombre": "Hielo zona electrificación (QICE+QHAIL+QGRAUP+QSNOW)", "unidades": "kg/kg",
     "cmap": "Blues", "func": calcular_intice_tmp},
    "hielo_animacion.gif")

In [ ]:
# QCLOUD + QRAIN
animar_variable(files_18_11_10,
    {"nombre": "Agua líquida (QCLOUD+QRAIN)", "unidades": "kg/kg",
     "cmap": "Blues", "keys": ["QCLOUD", "QRAIN"]},
    "agua_animacion.gif")

In [ ]:
# CTT
animar_variable(files_18_11_10,
    {"nombre": "Temperatura tope de nube", "unidades": "°C",
     "cmap": "RdBu_r", "keys": ["ctt"]},
    "ctt_animacion.gif")

In [ ]:
animar_variable(files_18_11_10,
    {"nombre": "LPI", "unidades": "J/kg",
     "cmap": "YlOrRd", "func": calcular_lpi},
    "lpi_animacion.gif")

In [ ]:
animar_variable(files_18_11_10,
    {"nombre": "Velocidad Vertical Máxima", "unidades": "m/s",
     "cmap": "RdBu_r", "keys": ["wa"]},
    "wmax_animacion.gif")

In [ ]:
OUTPUT_DIR = "/serafin/gcamilo/Trabajo_Final/dataset_6var"   # NUEVA carpeta
os.makedirs(OUTPUT_DIR, exist_ok=True)

CROP    = 1
N_PREV  = 5

VAR_CONFIGS = [
    {"nombre": "mdbz", "keys": ["mdbz"], "clip_min": 0,
     "getvar_kwargs": {"use_varint": True, "use_liqskin": True}},
    {"nombre": "hielo",     "func": calcular_intice_tmp},
    {"nombre": "agua",      "keys": ["QCLOUD", "QRAIN"]},
    {"nombre": "ctt",       "keys": ["ctt"]},
    {"nombre": "lpi",       "func": calcular_lpi},
    {"nombre": "wmax",      "keys": ["wa"]},
]
N_CANALES_POR_TS = len(VAR_CONFIGS)          # 6

In [ ]:
def extraer_canal(ds, var_config):
    """Extrae y procesa un canal desde un Dataset abierto."""
    
    # Caso función personalizada
    if "func" in var_config:
        resultado = var_config["func"](ds)
        return np.array(resultado)[CROP:-CROP, CROP:-CROP].astype(np.float32)
    
    resultado = None
    for key in var_config["keys"]:
        datos = to_np(getvar(ds, key))
        if datos.ndim == 3:
            datos = datos.max(axis=0)
        resultado = datos if resultado is None else resultado + datos
    if "clip_min" in var_config:
        resultado = np.maximum(resultado, var_config["clip_min"])
    return resultado[CROP:-CROP, CROP:-CROP].astype(np.float32)

def extraer_timestep(filepath):
    """Extrae todos los canales de un archivo."""
    ds = Dataset(filepath)
    canales = [extraer_canal(ds, vc) for vc in VAR_CONFIGS]
    ds.close()
    return np.stack(canales, axis=0)  # (N_CANALES_POR_TS, H, W)

def chequear_actividad(i, filepath):
    """Abre un archivo, revisa si hay rayos y devuelve su índice original."""
    try:
        ds = Dataset(filepath)
        # Asumo que to_np está importado de wrf
        lightdens = to_np(ds["LIGHTDENS"][0])
        ds.close()
        
        tiene_rayos = lightdens.max() > 0
        return i, tiene_rayos
    except Exception as e:
        print(f"Error leyendo {filepath}: {e}")
        return i, False

def t_archivo(fp):
    """Timestamp de un archivo wrfout a partir de su nombre."""
    m = re.search(r"wrfout_d\d+_(\d{4}-\d{2}-\d{2})_(\d{2}:\d{2}:\d{2})", os.path.basename(fp))
    return datetime.strptime(f"{m.group(1)} {m.group(2)}", "%Y-%m-%d %H:%M:%S")

def escanear_actividad(archivos):
    """Devuelve una lista de bool (uno por archivo): True si LIGHTDENS.max() > 0."""
    actividad = [False] * len(archivos)
    with ProcessPoolExecutor(max_workers=64) as executor:
        futuros = {
            executor.submit(chequear_actividad, i, f): i
            for i, f in enumerate(archivos)
        }
        for futuro in tqdm(as_completed(futuros), total=len(archivos), desc="Escaneando archivos", leave=False):
            idx, tiene_rayos = futuro.result()
            actividad[idx] = bool(tiene_rayos)
    return actividad

def tormentas_desde_actividad(actividad, n_prev=0):
    """
    Una tormenta es una racha de timesteps consecutivos con actividad (LIGHTDENS > 0);
    termina apenas la racha se corta. A cada tormenta se le agregan hasta n_prev timesteps
    previos sin actividad, SIN pisar a la tormenta anterior: si entre dos tormentas hay
    menos de n_prev timesteps inactivos (o si la tormenta arranca cerca del inicio de la
    corrida), se toman los que haya. Así cada timestep pertenece a una sola tormenta.

    Devuelve una lista de dicts con 'indices' (prev + activos), 'inicio_activo' y 'fin_activo'.
    """
    tormentas = []
    fin_previo = -1
    i, n = 0, len(actividad)
    while i < n:
        if actividad[i]:
            j = i
            while j + 1 < n and actividad[j + 1]:
                j += 1
            ini = max(0, i - n_prev, fin_previo + 1)
            tormentas.append({"indices": list(range(ini, j + 1)), "inicio_activo": i, "fin_activo": j})
            fin_previo = j
            i = j + 1
        else:
            i += 1
    return tormentas

def get_tormentas(archivos, n_prev=0):
    """Escanea los archivos de una corrida continua y devuelve sus tormentas."""
    return tormentas_desde_actividad(escanear_actividad(archivos), n_prev)

def fecha_archivo(filepath):
    match = re.search(r'wrfout_d\d+_(\d{4}-\d{2}-\d{2})_\d{2}:\d{2}:\d{2}', os.path.basename(filepath))
    return match.group(1) if match else None

def mes_de(fecha_str):
    return fecha_str[:7]

In [ ]:
# Rutas a las dos carpetas de salidas WRF (separadas por espacio en disco)
dir_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
dir_2019 = "../salidasWRF2019"

all_files = sorted(
    glob.glob(os.path.join(dir_2018, "wrfout_d01_*")) +
    glob.glob(os.path.join(dir_2019, "wrfout_d01_*"))
)
print(f"Total de archivos combinados: {len(all_files)}")

# Se agrupan los archivos en "corridas": tramos de timesteps consecutivos (separados exactamente por `paso`).
# Una tormenta puede abarcar varios días, así que se detecta sobre la corrida completa (no día por día).
ts_archivos  = {f: t_archivo(f) for f in all_files}
archivos_ord = sorted(all_files, key=ts_archivos.get)
paso = min(ts_archivos[b] - ts_archivos[a] for a, b in zip(archivos_ord[:-1], archivos_ord[1:]))
assert paso > timedelta(0), "Hay dos archivos con el mismo timestamp"

eventos = {}     # clave: fecha de inicio de la corrida -> lista ordenada de archivos
actual = [archivos_ord[0]]
for a, b in zip(archivos_ord[:-1], archivos_ord[1:]):
    if ts_archivos[b] - ts_archivos[a] == paso:
        actual.append(b)
    else:
        clave = f"{ts_archivos[actual[0]]:%Y-%m-%d}"
        assert clave not in eventos, f"Dos corridas empiezan el mismo día: {clave}"
        eventos[clave] = actual
        actual = [b]
clave = f"{ts_archivos[actual[0]]:%Y-%m-%d}"
assert clave not in eventos, f"Dos corridas empiezan el mismo día: {clave}"
eventos[clave] = actual

assert sum(len(v) for v in eventos.values()) == len(all_files)
print(f"Paso entre timesteps: {paso}")
print(f"\nCorridas continuas: {len(eventos)}")
for fecha in sorted(eventos.keys()):
    print(f"  {fecha}: {len(eventos[fecha])} archivos")


In [ ]:
NOMBRE = "wrfout_d01_2019-03-04_12:28:00"   # el .npz lo guardó con '-' en lugar de ':'
cands = glob.glob(os.path.join(dir_2018, NOMBRE)) + glob.glob(os.path.join(dir_2019, NOMBRE))
assert len(cands) == 1, cands
ruta = cands[0]

# Mismas definiciones que usaste para el dataset de 6 variables
vcs = {
    "AGUA": {"keys": ["QCLOUD", "QRAIN"]},
    "LPI":  {"func": calcular_lpi},
    "WMAX": {"keys": ["wa"]},
}
ds = Dataset(ruta)
extras = {n: extraer_canal(ds, vc) for n, vc in vcs.items()}   # ya recortadas con CROP
ds.close()

# Chequeo: MDBZ, HIELO y CTT recalculados deben coincidir con los guardados
z = dict(np.load("instante_1924.npz"))
chk = extraer_timestep(ruta)
print("coincide con lo guardado:", np.allclose(chk, z["crudo_X"][:3], atol=1e-4))
print({k: (v.shape, float(v.min()), float(v.max())) for k, v in extras.items()})

# Agregamos los extras al mismo archivo
for k, v in extras.items():
    z["extra_" + k] = v
np.savez_compressed("instante_1924.npz", **z)

In [ ]:
# -*- coding: utf-8 -*-
"""
Extracción de datos de los wrfout y del dataset para completar los
pendientes en rojo de las Secciones 3.1 y 3.5.4 de la monografía.

Qué responde cada función
-------------------------
estructura_simulaciones(eventos)
    * Cuántas simulaciones distintas hay dentro de los 17 eventos
      (agrupando por el atributo SIMULATION_START_DATE de cada
      wrfout), con fecha de inicio, de fin y horas de cada una.
    * Si algún atributo global (microfísica, paso de tiempo, etc.)
      cambia de un archivo a otro, o si la configuración es idéntica
      en todos (salvo las fechas de inicio).
variables_electricas(ruta)
    * Descripción y unidades que el propio modelo guarda para
      LIGHTDENS, LIGHT, INDUC, NONINDUC, las cargas por
      hidrometeoro, etc. Debería resolver qué mide LIGHTDENS.
serie_lightdens(eventos)
    * Para cada salida: suma de LIGHTDENS en todo el dominio (la
      serie que se compara con el LMA, Fig. 3.5), máximo, cantidad
      de píxeles activos y de píxeles con más de 150, y si los
      valores son enteros. Guarda un CSV (se puede reanudar).
fraccion_sobre_150(dir_dataset)
    * Qué fracción de los píxeles activos del dataset supera 150
      (y percentiles de los valores activos).

Uso en JupyterHub (Datasets.ipynb), una celda por paso
-------------------------------------------------------
    import sys; sys.path.append('../figuras')   # o copiar el archivo
    from extraer_metadatos_wrf import *

    # `eventos` ya existe si corriste la celda "Corridas continuas"
    # de Datasets.ipynb; si no:
    # eventos = armar_eventos()

    res_sim = estructura_simulaciones(eventos)     # unos minutos
    variables_electricas(eventos[sorted(eventos)[0]][0])
    serie = serie_lightdens(eventos)               # unos minutos
    fraccion_sobre_150('../dataset_temporal')

Después copiá y pegá en el chat lo que imprimen (o pasame los CSV
simulaciones_wrf.csv y serie_lightdens.csv).
"""
import glob
import json
import os
import re
from datetime import datetime

import numpy as np
import pandas as pd
from netCDF4 import Dataset

try:
    from tqdm.auto import tqdm
except ImportError:                      # sin tqdm, sin barra
    def tqdm(x, **kw):
        return x

PAT = re.compile(
    r"wrfout_d01_(\d{4}-\d{2}-\d{2})_(\d{2})[:\-](\d{2})[:\-](\d{2})")

# Atributos que cambian de una simulación a otra por construcción
# (todos describen el inicio de la simulación).
IGNORAR = {"START_DATE", "SIMULATION_START_DATE", "JULYR", "JULDAY",
           "GMT"}

ELEC_VARS = ["LIGHTDENS", "LIGHT", "LIGHTDIS", "INDUC", "NONINDUC",
             "ELECMAG", "ELECX", "ELECY", "ELECZ", "PC", "PCB",
             "SCW", "SCR", "SCI", "SCS", "SCH", "SCHL", "SCIONA",
             "SCTOT", "RSCGHIS_2D"]


def t_nombre(ruta):
    """Fecha y hora de la salida, tomadas del nombre del archivo."""
    m = PAT.search(os.path.basename(str(ruta)))
    return datetime.strptime(
        f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}",
        "%Y-%m-%d %H:%M:%S")


def armar_eventos(dirs=("~/jupyter/salidasWRF2018",
                        "../salidasWRF2019")):
    """Misma lógica que la celda de Datasets.ipynb: agrupa los
    wrfout en tramos de salidas consecutivas (un evento = fecha de
    inicio -> lista ordenada de archivos)."""
    archivos = []
    for d in dirs:
        archivos += glob.glob(os.path.join(os.path.expanduser(d),
                                           "wrfout_d01_*"))
    archivos.sort(key=t_nombre)
    ts = [t_nombre(f) for f in archivos]
    paso = min(b - a for a, b in zip(ts[:-1], ts[1:]))
    eventos, actual = {}, [archivos[0]]
    for f1, t0, t1 in zip(archivos[1:], ts[:-1], ts[1:]):
        if t1 - t0 == paso:
            actual.append(f1)
        else:
            eventos[f"{t_nombre(actual[0]):%Y-%m-%d}"] = actual
            actual = [f1]
    eventos[f"{t_nombre(actual[0]):%Y-%m-%d}"] = actual
    print(f"{sum(map(len, eventos.values()))} archivos, "
          f"{len(eventos)} eventos, paso {paso}")
    return eventos


# ---------------------------------------------------------------
# 1. Estructura de las simulaciones y consistencia de la config.
# ---------------------------------------------------------------
def _attrs(ruta):
    with Dataset(ruta) as ds:
        return {k: str(ds.getncattr(k)) for k in ds.ncattrs()}


def estructura_simulaciones(eventos, ruta_csv="simulaciones_wrf.csv",
                            cada=1):
    """Agrupa las salidas por SIMULATION_START_DATE dentro de cada
    evento y compara los atributos globales de todos los archivos
    con los del primero. `cada=k` lee un archivo de cada k (más
    rápido, pero las fronteras entre simulaciones quedan con
    precisión de k salidas); siempre se leen el primero y el último
    de cada evento."""
    ref, difieren, filas = None, {}, []
    for ev in tqdm(sorted(eventos), desc="Eventos"):
        lista = eventos[ev]
        for i, f in enumerate(lista):
            if cada > 1 and i % cada and i != len(lista) - 1:
                continue
            a = _attrs(f)
            if ref is None:
                ref = a
                with open("atributos_globales_ref.json", "w",
                          encoding="utf8") as fh:
                    json.dump({"archivo": os.path.basename(f),
                               "atributos": a}, fh, indent=1,
                              ensure_ascii=False)
            for k in set(a) | set(ref):
                if k not in IGNORAR and a.get(k) != ref.get(k):
                    d = difieren.setdefault(
                        (k, ref.get(k), a.get(k)),
                        [0, os.path.basename(f)])
                    d[0] += 1
            filas.append((ev, t_nombre(f),
                          a.get("SIMULATION_START_DATE")))
    df = pd.DataFrame(filas, columns=["evento", "t", "sim_inicio"])
    res = (df.groupby(["evento", "sim_inicio"], sort=True)
             .agg(salidas_leidas=("t", "size"), primera=("t", "min"),
                  ultima=("t", "max")).reset_index())
    res["horas"] = ((res.ultima - res.primera)
                    .dt.total_seconds() / 3600).round(2)
    res.to_csv(ruta_csv, index=False)
    print(res.to_string(index=False))
    print(f"\nSimulaciones distintas (por SIMULATION_START_DATE): "
          f"{res.sim_inicio.nunique()} en {res.evento.nunique()} "
          f"eventos. Guardado: {ruta_csv}")
    if difieren:
        print("\nAtributos globales que CAMBIAN respecto del primer "
              "archivo (sin contar fechas de inicio):")
        for (k, v0, v1), (n, ej) in difieren.items():
            print(f"  {k}: {v0} -> {v1}  ({n} archivos, p. ej. {ej})")
    else:
        print(f"\nTodos los atributos globales son idénticos al del "
              f"primer archivo (salvo las fechas de inicio) en los "
              f"{len(df)} archivos leídos.")
    return res


# ---------------------------------------------------------------
# 2. Metadatos de las variables eléctricas
# ---------------------------------------------------------------
def variables_electricas(ruta, nombres=ELEC_VARS):
    """Imprime dimensiones, unidades y descripción que el modelo
    guarda para cada variable eléctrica de un wrfout."""
    with Dataset(ruta) as ds:
        print("Archivo:", os.path.basename(str(ruta)))
        for n in nombres:
            if n not in ds.variables:
                print(f"{n:11s} -- no está en el archivo")
                continue
            v = ds.variables[n]
            print(f"{n:11s} dims={v.dimensions} "
                  f"unidades={getattr(v, 'units', '?')!r} "
                  f"descripción={getattr(v, 'description', '?')!r}")
        print("\nOtras variables con 'ELEC', 'LIGHT' o 'CHARGE' en "
              "el nombre:",
              [k for k in ds.variables
               if re.search("ELEC|LIGHT|CHARGE", k, re.I)
               and k not in nombres])


# ---------------------------------------------------------------
# 3. Serie de LIGHTDENS por salida (para comparar con el LMA)
# ---------------------------------------------------------------
COLS = ["evento", "t", "suma", "suma_clip150", "maximo", "n_activos",
        "n_mayor_150", "enteros"]


def _fila_lightdens(ev, f):
    with Dataset(f) as ds:
        y = np.ma.filled(ds.variables["LIGHTDENS"][0], 0.0)
    y = np.asarray(y, dtype=np.float64)
    act = y > 0
    return (ev, t_nombre(f), y.sum(), np.minimum(y, 150).sum(),
            y.max(), int(act.sum()), int((y > 150).sum()),
            bool(np.all(y[act] == np.round(y[act]))))


def serie_lightdens(eventos, ruta_csv="serie_lightdens.csv",
                    solo=None):
    """Una fila por salida con estadísticos de LIGHTDENS en todo el
    dominio (299x299, sin recortar). Guarda el CSV evento por evento
    y, si ya existe, saltea los eventos terminados. `solo` es una
    lista de claves de `eventos` para procesar solo esas."""
    previo = (pd.read_csv(ruta_csv, parse_dates=["t"])
              if os.path.exists(ruta_csv) else pd.DataFrame(columns=COLS))
    hechos = set(previo.evento.unique())
    for ev in tqdm(sorted(solo or eventos), desc="Eventos"):
        if ev in hechos:
            continue
        filas = [_fila_lightdens(ev, f) for f in eventos[ev]]
        nuevo = pd.DataFrame(filas, columns=COLS)
        nuevo.to_csv(ruta_csv, mode="a", index=False,
                     header=not os.path.exists(ruta_csv))
    df = pd.read_csv(ruta_csv, parse_dates=["t"])
    act = df[df.n_activos > 0]
    print(f"\n{len(df)} salidas, {len(act)} con actividad "
          f"({len(act) / len(df):.1%}).")
    print("Valores de LIGHTDENS enteros en todas las salidas "
          f"activas: {bool(act.enteros.all())}")
    print("Máximo en un píxel:", df.maximo.max(),
          "| máx. de la suma por salida:", round(df.suma.max(), 1))
    por_ev = (df.groupby("evento")
                .agg(suma_max=("suma", "max"),
                     suma_mediana_activas=(
                         "suma", lambda s: s[s > 0].median()),
                     pix_max=("maximo", "max"),
                     pix_activos=("n_activos", "sum"),
                     pix_mayor_150=("n_mayor_150", "sum")))
    print(por_ev.round(1).to_string())
    for dia in ("2018-12-14", "2019-02-09"):      # días con LMA
        d = df[df.t.dt.strftime("%Y-%m-%d") == dia]
        if len(d):
            print(f"\n{dia}: suma por salida (todo el dominio) "
                  f"máx {d.suma.max():.0f}, p95 "
                  f"{d.suma.quantile(.95):.0f}; recortada a 150: "
                  f"máx {d.suma_clip150.max():.0f}")
    return df


# ---------------------------------------------------------------
# 4. Fracción de píxeles activos que supera 150 (en el dataset)
# ---------------------------------------------------------------
def fraccion_sobre_150(dir_dataset="../dataset_temporal", umbral=150):
    """Lee solo `y` (valores crudos) de cada muestra del dataset."""
    archivos = sorted(glob.glob(os.path.join(dir_dataset,
                                             "wrfout_d01_*.npz")))
    activos, con_sobre = [], 0
    for f in tqdm(archivos, desc="Muestras"):
        with np.load(f) as z:
            y = np.squeeze(z["y"])
        v = y[y > 0]
        activos.append(v.astype(np.float32))
        con_sobre += bool((v > umbral).any())
    v = np.concatenate(activos)
    out = {
        "muestras": len(archivos),
        "pixeles_activos": int(v.size),
        "pixeles_sobre_umbral": int((v > umbral).sum()),
        "fraccion_pixeles": float((v > umbral).mean()),
        "muestras_con_algun_pixel_sobre_umbral": int(con_sobre),
        "percentiles_activos": {
            str(p): float(np.percentile(v, p))
            for p in (50, 90, 99, 99.9)},
        "maximo": float(v.max()),
    }
    print(json.dumps(out, indent=1, ensure_ascii=False))
    return out

In [ ]:
# -*- coding: utf-8 -*-
"""
Serie de descargas cada 2 minutos: LMA y WRF-ELEC (Fig. 4.x).

El LMA (RELAMPAGO) solo registra descargas a menos de 100 km de su
centro (-31,7 / -64,1); el dominio del WRF abarca unos 600 km de
lado. Para que la comparación sea directa, la serie del WRF se
calcula (a) en todo el dominio y (b) solo dentro de ese radio.

Es autocontenido (no importa nada propio). Uso en JupyterHub: pegar
todo el archivo en una celda de Datasets.ipynb, donde ya existe
`eventos` (celda "Corridas continuas"), y agregar al final:

    serie_wrf_en_radio(eventos)      # escribe serie_wrf_radio_25ene.csv

Después pasame serie_wrf_radio_25ene.csv.
"""
import multiprocessing as mp
import numpy as np
import pandas as pd
from netCDF4 import Dataset

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw):
        return x

import os
import re
from datetime import datetime

PAT = re.compile(
    r"wrfout_d01_(\d{4}-\d{2}-\d{2})_(\d{2})[:\-](\d{2})[:\-](\d{2})")


def t_nombre(ruta):
    """Fecha y hora de la salida, tomadas del nombre del archivo."""
    m = PAT.search(os.path.basename(str(ruta)))
    return datetime.strptime(
        f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}",
        "%Y-%m-%d %H:%M:%S")


LAT0, LON0, RADIO_KM = -31.7, -64.1, 100.0     # centro y radio del LMA
DIAS = ("2019-01-25",)                          # día de mayor actividad del LMA
N_PROC = 16


def _dist_km(lat, lon):
    r = 6371.0
    p1, p2 = np.radians(lat), np.radians(LAT0)
    dl = np.radians(lon - LON0)
    a = (np.sin((p2 - p1) / 2) ** 2 +
         np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2)
    return 2 * r * np.arcsin(np.sqrt(a))


def _leer(ds, nombre):
    if nombre not in ds.variables:
        return None
    v = np.array(ds.variables[nombre][:], dtype="float64")
    return np.squeeze(v)          # (Time, y, x) con un solo tiempo


def _fila(f):
    with Dataset(f) as ds:
        lat = _leer(ds, "XLAT")
        lon = _leer(ds, "XLONG")
        lg = _leer(ds, "LIGHT")
        ld = _leer(ds, "LIGHTDENS")
    # las variables escalonadas pueden tener 300 en alguna dimensión
    n = min(lat.shape[0], lg.shape[0])
    m = min(lat.shape[1], lg.shape[1])
    lat, lon, lg, ld = (a[:n, :m] for a in (lat, lon, lg, ld))
    dentro = _dist_km(lat, lon) <= RADIO_KM
    return (t_nombre(f), lg.sum(), lg[dentro].sum(),
            ld.sum(), ld[dentro].sum(),
            np.minimum(ld, 150)[dentro].sum(), int(dentro.sum()))


def serie_wrf_en_radio(eventos, ruta_csv="serie_wrf_radio_25ene.csv",
                       dias=DIAS, n_proc=N_PROC):
    archivos = sorted({f for fs in eventos.values() for f in fs
                       if f"{t_nombre(f):%Y-%m-%d}" in dias},
                      key=t_nombre)
    print(len(archivos), "salidas en", dias)
    with mp.get_context("fork").Pool(n_proc) as pool:
        filas = list(tqdm(pool.imap(_fila, archivos, chunksize=4),
                          total=len(archivos)))
    df = pd.DataFrame(filas, columns=[
        "t", "light_dominio", "light_radio", "lightdens_dominio",
        "lightdens_radio", "lightdens_radio_sat150", "n_pix_radio"])
    df.to_csv(ruta_csv, index=False)
    for dia in dias:
        d = df[df.t.dt.strftime("%Y-%m-%d") == dia]
        if len(d):
            print(dia, "| máx LIGHT dominio", d.light_dominio.max(),
                  "| máx LIGHT en el radio", d.light_radio.max(),
                  "| píxeles en el radio", int(d.n_pix_radio.iloc[0]))
    return df

serie_wrf_en_radio(eventos)

In [ ]:
tormentas   = []     # una entrada por tormenta (racha de LIGHTDENS > 0 + hasta N_PREV timesteps previos)
all_samples = {}     # corrida -> índices (ordenados) de todos sus samples filtrados
total_global = 0
for fecha in tqdm(sorted(eventos.keys()), desc="Corridas"):
    lista = get_tormentas(eventos[fecha], n_prev=N_PREV)
    todos = [i for t in lista for i in t["indices"]]
    assert len(todos) == len(set(todos)), f"Samples repetidos entre tormentas de la corrida {fecha}"
    all_samples[fecha] = sorted(todos)
    total_global += len(todos)
    for t in lista:
        t0 = t_archivo(eventos[fecha][t["inicio_activo"]])
        tormentas.append({
            "id": f"{t0:%Y-%m-%d_%H%M}",          # momento en que arranca la actividad
            "corrida": fecha,
            "indices": t["indices"],
            "n_activos": t["fin_activo"] - t["inicio_activo"] + 1,
        })

assert len({t["id"] for t in tormentas}) == len(tormentas)
assert sum(len(t["indices"]) for t in tormentas) == total_global

print(f"\nTotal global de samples: {total_global} | tormentas: {len(tormentas)}\n")
for fecha in sorted(all_samples.keys()):
    ts = [t for t in tormentas if t["corrida"] == fecha]
    print(f"  corrida {fecha}: {len(all_samples[fecha])} samples en {len(ts)} tormenta(s)")
    for t in ts:
        print(f"      {t['id']}: {len(t['indices'])} samples ({t['n_activos']} activos + {len(t['indices']) - t['n_activos']} previos)")


In [ ]:
# ==========================================================
# Selección de tormentas para el conjunto de validación
# ==========================================================
# Unidad de selección: la tormenta (racha de LIGHTDENS > 0 + sus timesteps previos), tal como sale de `tormentas`.
# Cada tormenta se asigna al mes en el que arranca su actividad.
#
# Criterios, en orden de prioridad:
#   1. Cantidad de samples de val ≈ FRAC_VAL del total (train = el resto).
#   2. Variabilidad: 1 tormenta por mes (nov y dic 2018; ene, feb y mar 2019).
#      Si con eso no se llega al objetivo dentro de la ventana, se cae al esquema 2:
#      1 tormenta de 2018 (nov o dic) + 1 por mes de 2019.
#   Entre las combinaciones válidas se elige la que queda más cerca del objetivo.
#
# Guarda anti-fuga: dos tormentas separadas por poco tiempo tienen samples casi pegados. Si una va a val
# y la otra a train, se filtra información entre conjuntos. Las tormentas que tienen otra a menos de
# GAP_MIN_MIN minutos no se usan como candidatas.

FRAC_VAL          = 0.20   # objetivo: fracción del total de samples que va a val
TOL_ABAJO         = 0.02   # cuánto por DEBAJO del objetivo se acepta (fracción del objetivo). 0 = nunca menos
TOL_ARRIBA        = 0.02   # cuánto por ARRIBA del objetivo se acepta (fracción del objetivo)
GAP_MIN_MIN       = 60     # separación mínima (min) entre un sample de val y uno de train
EVITAR_VECINAS    = True   # descartar como candidatas las tormentas con otra a menos de GAP_MIN_MIN

ESQUEMAS = [
    [["2018-11"], ["2018-12"], ["2019-01"], ["2019-02"], ["2019-03"]],   # 1 tormenta por mes
    [["2018-11", "2018-12"], ["2019-01"], ["2019-02"], ["2019-03"]],     # 1 de 2018 + 1 por mes de 2019
]

info = {}
for t in tormentas:
    ts = [t_archivo(eventos[t["corrida"]][i]) for i in t["indices"]]
    info[t["id"]] = {"n": len(ts), "t0": min(ts), "t1": max(ts), "mes": t["id"][:7], "corrida": t["corrida"]}

total    = sum(v["n"] for v in info.values())
objetivo = FRAC_VAL * total
lim_inf  = objetivo * (1 - TOL_ABAJO)
lim_sup  = objetivo * (1 + TOL_ARRIBA)

# --- Tormentas cercanas en el tiempo ---
gap_min = timedelta(minutes=GAP_MIN_MIN)
orden   = sorted(info, key=lambda e: info[e]["t0"])
vecinas = {e: set() for e in info}
for a, b in zip(orden[:-1], orden[1:]):
    if info[b]["t0"] - info[a]["t1"] < gap_min:
        vecinas[a].add(b)
        vecinas[b].add(a)

candidatas  = [e for e in orden if not (EVITAR_VECINAS and vecinas[e])]
descartadas = [e for e in orden if e not in candidatas]
print(f"Candidatas: {len(candidatas)} de {len(info)} tormentas")
if descartadas:
    print(f"Descartadas por tener otra tormenta a menos de {GAP_MIN_MIN} min:")
    for e in descartadas:
        print(f"  {e}  ↔  {sorted(vecinas[e])}")
print()

# --- Búsqueda (programación dinámica sobre la suma de samples: una tormenta por grupo) ---
def buscar(esquema):
    grupos = [[e for e in candidatas if info[e]["mes"] in g] for g in esquema]
    if any(len(g) == 0 for g in grupos):
        return []
    alcanzables = {0: ()}
    for g in grupos:
        nuevo = {}
        for s, c in alcanzables.items():
            for e in g:
                nuevo.setdefault(s + info[e]["n"], c + (e,))
        alcanzables = nuevo
    return sorted((abs(s - objetivo), s, c) for s, c in alcanzables.items() if lim_inf <= s <= lim_sup)

for i_esquema, esquema in enumerate(ESQUEMAS, 1):
    validas = buscar(esquema)
    if validas:
        break
else:
    raise ValueError("Ninguna combinación cumple los criterios: ajustá TOL_ARRIBA/TOL_ABAJO/GAP_MIN_MIN o elegí val_tormentas a mano.")

_, n_val, combo = validas[0]
val_tormentas = sorted(combo)
# val_tormentas = [...]   # <- descomentar para forzar una selección manual (ids de `tormentas`)

# --- Reporte ---
print(f"Total de samples: {total} | objetivo val ({FRAC_VAL:.0%}): {objetivo:.0f} | ventana aceptada: [{lim_inf:.0f}, {lim_sup:.0f}]")
print(f"Esquema usado: {i_esquema}\n")
print("Tormentas elegidas para validación:")
for e in val_tormentas:
    print(f"  {e}  ({info[e]['mes']}): {info[e]['n']} samples | {info[e]['t0']:%Y-%m-%d %H:%M} → {info[e]['t1']:%Y-%m-%d %H:%M}")

n_val = sum(info[e]["n"] for e in val_tormentas)
print(f"\nVal:   {n_val} samples ({100 * n_val / total:.2f}%)")
print(f"Train: {total - n_val} samples ({100 * (total - n_val) / total:.2f}%)")

print("\nSiguientes mejores combinaciones del mismo esquema:")
for _, s, c in validas[1:6]:
    print(f"  {s} ({100 * s / total:.2f}%): {list(c)}")

# --- Chequeo de separación val/train (también sirve si se fuerza val_tormentas a mano) ---
train_tormentas = [e for e in info if e not in val_tormentas]
conflictos = [(v, t) for v in val_tormentas for t in train_tormentas if t in vecinas[v]]
print(f"\nChequeo de separación val/train (≥ {GAP_MIN_MIN} min):",
      "OK" if not conflictos else f"¡ATENCIÓN! {conflictos}")


In [ ]:
# ==========================================================
# Tasa de cambio de los extremos de cada variable, por tormenta
# (max para mdbz/hielo, min para ctt), entre timesteps consecutivos
# ==========================================================

EXTREMO_POR_VARIABLE = {"mdbz": "max", "hielo": "max", "ctt": "min"}

def extraer_extremo_archivo(archivo):
    """Para un archivo, devuelve el extremo (max o min según corresponda) de cada variable."""
    ds = Dataset(archivo)
    resultado = {}
    for vc in VAR_CONFIGS:
        nombre = vc["nombre"]
        if nombre not in EXTREMO_POR_VARIABLE:
            continue
        canal = extraer_canal(ds, vc)
        stat = EXTREMO_POR_VARIABLE[nombre]
        resultado[nombre] = float(canal.max()) if stat == "max" else float(canal.min())
    ds.close()
    return resultado

def calcular_tasa_cambio_tormenta(t, eventos):
    """
    Para una tormenta (dict de `tormentas`), calcula:
      - 'extremos': la serie temporal del extremo de cada variable (uno por timestep)
      - 'tasa': la diferencia cruda entre timesteps consecutivos (valor[i+1] - valor[i])
    También devuelve 'n_prev': cantidad de timesteps previos sin actividad al inicio
    de la tormenta (antes de que arranque LIGHTDENS > 0), para poder separarlos al graficar.
    """
    archivos_t = [eventos[t["corrida"]][idx] for idx in t["indices"]]
    n_prev = len(t["indices"]) - t["n_activos"]

    with ProcessPoolExecutor(max_workers=16) as executor:
        extremos_por_ts = list(executor.map(extraer_extremo_archivo, archivos_t))

    resultado = {"n_prev": n_prev}
    for nombre in EXTREMO_POR_VARIABLE:
        serie = [e[nombre] for e in extremos_por_ts]
        tasa = [serie[i + 1] - serie[i] for i in range(len(serie) - 1)]
        resultado[nombre] = {"extremos": serie, "tasa": tasa}
    return resultado

def graficar_tasa_cambio_tormenta(t, eventos, guardar=True, ruta_base='../images'):
    """
    Grafica, para una tormenta, la tasa de cambio de cada variable a lo largo del tiempo.
    Sombrea en gris los timesteps previos (sin actividad eléctrica) y marca con una
    línea vertical el inicio de la actividad (LIGHTDENS > 0).
    """
    resultado = calcular_tasa_cambio_tormenta(t, eventos)
    n_prev = resultado["n_prev"]

    fig, axes = plt.subplots(len(EXTREMO_POR_VARIABLE), 1, figsize=(9, 3 * len(EXTREMO_POR_VARIABLE)), sharex=True)
    for ax, (nombre, stat) in zip(axes, EXTREMO_POR_VARIABLE.items()):
        tasa = resultado[nombre]["tasa"]
        xs = range(1, len(tasa) + 1)

        if n_prev > 0:
            ax.axvspan(0.5, n_prev + 0.5, color='gray', alpha=0.15, label='Previos (sin actividad)')
        ax.axvline(n_prev + 0.5, color='red', linestyle='--', alpha=0.6, linewidth=1, label='Inicio actividad')

        ax.plot(xs, tasa, 'o-', markersize=4, color='steelblue')
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_ylabel(f"Δ {nombre} ({stat})")
        ax.grid(alpha=0.3)

    axes[0].legend(loc='upper right', fontsize=8)
    axes[-1].set_xlabel("Timestep")
    fig.suptitle(f"Tasa de cambio de extremos — Tormenta {t['id']}")
    plt.tight_layout()
    
    if guardar:
        import os
        os.makedirs(ruta_base, exist_ok=True)  # Se asegura de que la carpeta exista
        ruta_guardado = f"{ruta_base}/tasa_cambio_tormenta_{t['id']}.png"
        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
        print(f"Gráfico guardado en: {ruta_guardado}")

    plt.show()

    return resultado

tormentas_val = [t for t in tormentas if t["id"] in val_tormentas]

for t in tormentas_val:
    graficar_tasa_cambio_tormenta(t, eventos)

In [ ]:
# Rango de valores de cada variable para las tormentas filtradas

def procesar_un_archivo(archivo):
    # Diccionario local de este núcleo
    stats_local = {vc["nombre"]: {"min": float("inf"), "max": float("-inf")} for vc in VAR_CONFIGS}
    try:
        ds = Dataset(archivo)
        for vc in VAR_CONFIGS:
            canal = extraer_canal(ds, vc)
            nombre = vc["nombre"]
            stats_local[nombre]["min"] = float(canal.min())
            stats_local[nombre]["max"] = float(canal.max())
            del canal
        ds.close()
        del ds
    except Exception as e:
        print(f"Error procesando {archivo}: {e}")
        return None

    return stats_local

output_path = os.path.expanduser("../dataset/rangos_variables_eventos.csv")

# 1. Chequear si el archivo ya existe para reanudar sin repetir eventos
eventos_procesados = set()
if os.path.exists(output_path):
    try:
        df_existente = pd.read_csv(output_path)
        if "fecha" in df_existente.columns:
            eventos_procesados = set(df_existente["fecha"].unique())
            print(f"Reanudando ejecución. Eventos ya procesados: {len(eventos_procesados)}")
    except Exception as e:
        print(f"Atención: No se pudo leer el CSV existente ({e}), se creará uno nuevo.")

# 2. Bucle principal con guardado incremental
for fecha in tqdm(sorted(all_samples.keys()), desc="Eventos"):

    # Si la fecha ya se procesó en una corrida anterior, se saltea
    if fecha in eventos_procesados:
        continue

    archivos = eventos[fecha]
    indices = all_samples[fecha]
    rutas_archivos = [archivos[idx] for idx in indices]
    stats_evento = {vc["nombre"]: {"min": float("inf"), "max": float("-inf")} for vc in VAR_CONFIGS}

    with ProcessPoolExecutor(max_workers=64) as executor:
        futuros = [executor.submit(procesar_un_archivo, ruta) for ruta in rutas_archivos]

        for futuro in as_completed(futuros):
            resultado = futuro.result()
            if resultado is None:
                continue
            for nombre, vals in resultado.items():
                stats_evento[nombre]["min"] = min(stats_evento[nombre]["min"], vals["min"])
                stats_evento[nombre]["max"] = max(stats_evento[nombre]["max"], vals["max"])

    # Guardado incremental: una fila por variable, con la fecha del evento
    filas = [
        {"fecha": fecha, "variable": nombre, "min": vals["min"], "max": vals["max"]}
        for nombre, vals in stats_evento.items()
    ]
    df_nuevo = pd.DataFrame(filas)
    header = not os.path.exists(output_path)
    df_nuevo.to_csv(output_path, mode="a", header=header, index=False)

In [ ]:
output_path = os.path.expanduser("../dataset/rangos_variables_eventos.csv")
df_rangos = pd.read_csv(output_path)

# Agrupar por variable y calcular el mínimo absoluto y máximo absoluto
df_globales = df_rangos.groupby("variable").agg(
    min_global=("min", "min"),
    max_global=("max", "max")
).reset_index()

print("--- RANGOS GLOBALES POR VARIABLE ---")
display(df_globales)

# Fase 4: Creación del Dataset

In [ ]:
H = W = 299 - 2 * CROP  # 297

# val_tormentas viene de la celda de selección de tormentas (arriba).
# Cada sample pertenece a una sola tormenta, así que val y train se arman sin solapamiento.
val_ids = set(val_tormentas)
assert val_ids <= {t["id"] for t in tormentas}, "val_tormentas tiene ids que no existen"

val_files, train_files = [], []
for t in tormentas:
    archivos_t = [eventos[t["corrida"]][idx] for idx in t["indices"]]
    (val_files if t["id"] in val_ids else train_files).extend(archivos_t)

# --- Chequeos de integridad: sin repetidos y sin solapamiento entre conjuntos ---
assert len(set(train_files)) == len(train_files), "Hay archivos repetidos en train"
assert len(set(val_files)) == len(val_files), "Hay archivos repetidos en val"
assert set(train_files).isdisjoint(val_files), "Hay archivos que están en train Y en val"
assert len(train_files) + len(val_files) == total_global, "train + val no suma el total filtrado"
_nombres = [os.path.basename(f) for f in train_files + val_files]
assert len(set(_nombres)) == len(_nombres), "Nombres de archivo repetidos (se pisarían al guardar los .npz)"

print(f'Total archivos de entrenamiento: {len(train_files)}')
print(f'Total archivos de validación: {len(val_files)}')
print(f'Total general: {len(train_files) + len(val_files)}')
print(f'Val: {100 * len(val_files) / (len(train_files) + len(val_files)):.2f}%')


In [ ]:
UMBRAL_LIGHTDENS = 150  # valores mayores se condensan a este tope

def procesar_archivo(filepath, split, output_dir):
    ds = Dataset(filepath)
    lightdens = to_np(ds["LIGHTDENS"][0])
    y = lightdens[CROP:-CROP, CROP:-CROP].astype(np.float32)
    ds.close()

    # Condensamos: todo valor > 150 pasa a valer 150
    y = np.minimum(y, UMBRAL_LIGHTDENS)

    # Extraer X
    X = extraer_timestep(filepath)

    # Guardado comprimido
    nombre = os.path.basename(filepath).replace(":", "-")
    save_path = os.path.join(output_dir, split, f"{nombre}.npz")
    np.savez_compressed(save_path, X=X, y=y)
    return split

# Limpieza antes de regenerar (para no dejar huérfanos si cambió el split)
import shutil
for split in ["train", "val"]:
    split_dir = os.path.join(OUTPUT_DIR, split)
    if os.path.exists(split_dir):
        shutil.rmtree(split_dir)
    os.makedirs(split_dir, exist_ok=True)

tareas = [(f, "train", OUTPUT_DIR) for f in train_files]

n_train, n_val = 0, 0
print(f"Lanzando procesamiento en paralelo para {len(tareas)} archivos...")

with ProcessPoolExecutor(max_workers=16) as executor:
    futuros = [executor.submit(procesar_archivo, *tarea) for tarea in tareas]
    for futuro in tqdm(as_completed(futuros), total=len(tareas), desc="Procesando"):
        split_result = futuro.result()
        if split_result == "train":
            n_train += 1
        elif split_result == "val":
            n_val += 1

print(f"\nDataset guardado:")
print(f"  Train: {n_train} muestras")
print(f"  Val:   {n_val} muestras")
print(f"Shape X: ({N_CANALES_POR_TS}, {H}, {W})")
print(f"Shape y: ({H}, {W})")

In [ ]:
# Celda para pegar en Datasets.ipynb DESPUÉS de regenerar el dataset
# con las seis variables en una carpeta aparte (ver pasos en el chat:
# OUTPUT_DIR distinto, VAR_CONFIGS con las seis variables).
# Calcula, sobre el conjunto de entrenamiento y recorriéndolo dos
# veces:
#  - las matrices de correlación de Pearson (todos los píxeles y solo
#    los píxeles con LIGHTDENS > 0), con LIGHTDENS tal como está
#    guardada (saturada en 150, sin transformar);
#  - los histogramas de cada variable, separados por LIGHTDENS > 0 y
#    LIGHTDENS = 0.
# Guarda todo en analisis_6var.npz; ese archivo va en
# figuras/datos/ de la monografía.
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor

DIR_TRAIN = f"{OUTPUT_DIR}/train"   # OUTPUT_DIR = carpeta de 6 variables
N_BINS = 200
# Potencia del espaciado de los bordes de los intervalos: 3 deja
# intervalos finos cerca del mínimo, donde está casi toda la masa de
# MDBZ, HIELO, AGUA, LPI y WMAX; CTT (bimodal) usa espaciado uniforme.
POTENCIA = {"CTT": 1}
files = sorted(Path(DIR_TRAIN).glob("*.npz"))
nombres = [v["nombre"].upper() for v in VAR_CONFIGS]
n = len(nombres)

def _minmax(f):
    X = np.load(f)["X"]
    assert np.isfinite(X).all(), f"NaN o inf en {f}"
    return X.min(axis=(1, 2)), X.max(axis=(1, 2))

def _pasada(args):
    f, bins = args
    d = np.load(f)
    X = d["X"].astype(np.float64)
    y = d["y"].astype(np.float64).ravel()
    Xf = X.reshape(n, -1).T
    C = np.hstack([Xf, y[:, None]])
    con = y > 0
    st, h = [], np.zeros((2, n, N_BINS))
    for k, sel in enumerate((con, ~con)):
        c = C[sel]
        st.append((len(c), c.sum(0), c.T @ c))
        for i in range(n):
            h[k, i] = np.histogram(Xf[sel, i], bins=bins[i])[0]
    return st, h

def _pearson(nn, s, p):
    cov = p - np.outer(s, s) / nn
    sd = np.sqrt(np.outer(np.diag(cov), np.diag(cov)))
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.nan_to_num(cov / sd)

with ProcessPoolExecutor(16) as ex:
    mm = list(tqdm(ex.map(_minmax, files), total=len(files),
                   desc="Mínimos y máximos"))
mins = np.min([m[0] for m in mm], axis=0)
maxs = np.max([m[1] for m in mm], axis=0)
bins = np.array([a + (b - a) * np.linspace(0, 1, N_BINS + 1)
                 ** POTENCIA.get(nom, 3)
                 for nom, a, b in zip(nombres, mins, maxs)])

tot = [[0, 0, 0], [0, 0, 0]]
H = np.zeros((2, n, N_BINS))
with ProcessPoolExecutor(16) as ex:
    for st, h in tqdm(ex.map(_pasada, [(f, bins) for f in files]),
                      total=len(files), desc="Estadísticos"):
        H += h
        for k in range(2):
            for j in range(3):
                tot[k][j] = tot[k][j] + st[k][j]

corr_con = _pearson(*tot[0])
corr_todos = _pearson(tot[0][0] + tot[1][0], tot[0][1] + tot[1][1],
                      tot[0][2] + tot[1][2])
np.savez("analisis_6var.npz", vars=np.array(nombres + ["LIGHTDENS"]),
         corr_todos=corr_todos, corr_con=corr_con,
         bins=bins, h_con=H[0], h_sin=H[1],
         n_muestras=len(files))
print("Listo: analisis_6var.npz", len(files), "muestras")

In [ ]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

def obtener_estadisticos_locales(filepath):
    """
    Función que ejecuta cada worker: 
    Abre un archivo, extrae los min/max de X e y, y los devuelve.
    """
    data = np.load(filepath)
    
    # 1. Extraemos y calculamos min/max espaciales de X (3 canales)
    X = data["X"]
    min_X_local = X.min(axis=(1, 2))
    max_X_local = X.max(axis=(1, 2))
    
    # 2. Extraemos, aplicamos log y calculamos min/max de y
    y_log = np.log(data["y"].astype(np.float64) + 0.1)
    min_y_local = y_log.min()
    max_y_local = y_log.max()
    
    return min_X_local, max_X_local, min_y_local, max_y_local

def calcular_estadisticos_globales_paralelo(split_dir, n_canales=3, max_workers=16):
    """Orquesta el cálculo en paralelo y consolida los globales."""
    files = sorted(Path(split_dir).glob("*.npz"))
    
    # Inicializamos los globales con valores extremos
    min_X_global = np.full(n_canales, np.inf, dtype=np.float64)
    max_X_global = np.full(n_canales, -np.inf, dtype=np.float64)
    min_y_global = float("inf")
    max_y_global = float("-inf")
    
    print(f"Lanzando cálculo en paralelo para {len(files)} archivos con {max_workers} workers...")
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Enviamos todas las tareas
        futuros = [executor.submit(obtener_estadisticos_locales, f) for f in files]
        
        # Procesamos a medida que terminan
        for futuro in tqdm(as_completed(futuros), total=len(files), desc="Calculando Estadísticos"):
            min_X_loc, max_X_loc, min_y_loc, max_y_loc = futuro.result()
            
            # Actualizamos globales comparando con los locales que acaban de llegar
            min_X_global = np.minimum(min_X_global, min_X_loc)
            max_X_global = np.maximum(max_X_global, max_X_loc)
            
            min_y_global = min(min_y_global, min_y_loc)
            max_y_global = max(max_y_global, max_y_loc)
            
    return min_X_global, max_X_global, min_y_global, max_y_global

# SOLO TRAIN
min_X, max_X, min_y, max_y = calcular_estadisticos_globales_paralelo(f"{OUTPUT_DIR}/train", n_canales=3, max_workers=16)

# Mostramos resultados de X
print("\n--- ESTADÍSTICOS DE ENTRADA (X) [MIN-MAX] ---")
for i, config in enumerate(VAR_CONFIGS):
    nombre = config["nombre"]
    print(f"Canal {i} | {nombre.upper()}:")
    print(f"  Mínimo global: {min_X[i]:>12.4f}")
    print(f"  Máximo global: {max_X[i]:>12.4f}\n")

# Mostramos resultados de y
print(f"--- ESTADÍSTICOS DE SALIDA (y) ---")
print(f"log(LIGHTDENS+0.1) — min: {min_y:.4f}, max: {max_y:.4f}\n")

# Guardamos los estadísticos
stats_path = f"{OUTPUT_DIR}/stats.npz"
np.savez(stats_path, min_X=min_X, max_X=max_X, min_y=min_y, max_y=max_y)
print(f"¡Archivos guardados en {stats_path}!")

In [ ]:
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

def obtener_min_max_lightdens_local(filepath):
    """
    Función worker: Extrae el mínimo y máximo real de la variable y (LIGHTDENS).
    """
    data = np.load(filepath)
    y = data["y"]
    # Convertimos a float nativo de Python para que el JSON lo pueda leer sin chistar
    return float(y.min()), float(y.max())

def analizar_lightdens_paralelo(split_dir, max_workers=16):
    """
    Orquesta el cálculo en paralelo para un directorio específico.
    """
    files = sorted(Path(split_dir).glob("*.npz"))
    
    min_y_global = float("inf")
    max_y_global = float("-inf")
    
    print(f"Lanzando cálculo para {len(files)} archivos en '{split_dir}'...")
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futuros = [executor.submit(obtener_min_max_lightdens_local, f) for f in files]
        
        desc_tqdm = f"Analizando {Path(split_dir).name.upper()}"
        
        for futuro in tqdm(as_completed(futuros), total=len(files), desc=desc_tqdm):
            min_y_loc, max_y_loc = futuro.result()
            
            min_y_global = min(min_y_global, min_y_loc)
            max_y_global = max(max_y_global, max_y_loc)
            
    return min_y_global, max_y_global


# --- EJECUCIÓN PARA TRAIN Y VAL ---
resultados_globales = {}

for split in ["train", "val"]:
    split_dir = f"{OUTPUT_DIR}/{split}"
    
    # Calculamos
    min_y, max_y = analizar_lightdens_paralelo(split_dir, max_workers=16)
    
    # Guardamos en el diccionario
    resultados_globales[split] = {
        "minimo_global": min_y,
        "maximo_global": max_y
    }
    
    # Mostramos los resultados por pantalla
    print(f"\n--- ESTADÍSTICOS LIGHTDENS ({split.upper()}) ---")
    print(f"  Mínimo global: {min_y}")
    print(f"  Máximo global: {max_y}\n")

# Guardamos todo en un único archivo JSON en el directorio principal del dataset
stats_path_json = f"{OUTPUT_DIR}/lightdens_limites.json"

with open(stats_path_json, "w") as f:
    json.dump(resultados_globales, f, indent=4)

print("-" * 60)
print(f"¡Listo! Estadísticos consolidados y guardados en formato legible en:")
print(stats_path_json)

In [ ]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt

# ==========================================
# FASE 1: WORKER DE ESTADÍSTICOS GLOBALES
# ==========================================
def worker_stats(filepath):
    data = np.load(filepath)
    X = data["X"].astype(np.float64)
    y_log = np.log(data["y"].astype(np.float64) + 0.1)
    
    suma = X.sum(axis=(1, 2))
    suma_c2 = (X**2).sum(axis=(1, 2))
    n_pixels = X.shape[1] * X.shape[2]
    
    mins = X.min(axis=(1, 2))
    maxs = X.max(axis=(1, 2))
    
    min_y = y_log.min()
    max_y = y_log.max()
    
    return suma, suma_c2, n_pixels, mins, maxs, min_y, max_y

# ==========================================
# FASE 2: WORKER DE HISTOGRAMAS
# ==========================================
def worker_histogramas(args):
    filepath, medias, stds, mins, maxs, z_bins, mm_bins, y_bins, n_vars = args
    data = np.load(filepath)
    X = data["X"].astype(np.float64)
    y_log = np.log(data["y"].flatten() + 0.1)
    
    mask_torm = (data["y"].flatten() > 0)
    mask_notorm = ~mask_torm
    
    # Normalizaciones
    X_z = (X - medias[:, None, None]) / (stds[:, None, None] + 1e-8)
    X_mm = (X - mins[:, None, None]) / (maxs[:, None, None] - mins[:, None, None] + 1e-8)
    
    # Inicializamos contadores locales
    counts_z_torm = np.zeros((n_vars, len(z_bins)-1))
    counts_z_notorm = np.zeros((n_vars, len(z_bins)-1))
    counts_mm_torm = np.zeros((n_vars, len(mm_bins)-1))
    counts_mm_notorm = np.zeros((n_vars, len(mm_bins)-1))
    
    for i in range(n_vars):
        flat_z = X_z[i].flatten()
        flat_mm = X_mm[i].flatten()
        
        counts_z_torm[i], _ = np.histogram(flat_z[mask_torm], bins=z_bins)
        counts_z_notorm[i], _ = np.histogram(flat_z[mask_notorm], bins=z_bins)
        counts_mm_torm[i], _ = np.histogram(flat_mm[mask_torm], bins=mm_bins)
        counts_mm_notorm[i], _ = np.histogram(flat_mm[mask_notorm], bins=mm_bins)
        
    counts_y_torm, _ = np.histogram(y_log[mask_torm], bins=y_bins)
    counts_y_notorm, _ = np.histogram(y_log[mask_notorm], bins=y_bins)
    
    return counts_z_torm, counts_z_notorm, counts_mm_torm, counts_mm_notorm, counts_y_torm, counts_y_notorm

# ==========================================
# FUNCIÓN PRINCIPAL ORQUESTADORA
# ==========================================
def plot_comparacion_distribuciones_paralelo(split_dir, var_configs, max_workers=16):
    files = sorted(Path(split_dir).glob("*.npz"))
    n_vars = len(var_configs)
    
    # --- PASO 1: Estadísticos globales en paralelo ---
    suma = np.zeros(n_vars, dtype=np.float64)
    suma_c2 = np.zeros(n_vars, dtype=np.float64)
    n_pixels_total = 0
    mins_glob = np.full(n_vars, np.inf)
    maxs_glob = np.full(n_vars, -np.inf)
    min_y_glob, max_y_glob = float('inf'), float('-inf')
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futs = [executor.submit(worker_stats, f) for f in files]
        for fut in tqdm(as_completed(futs), total=len(files), desc="Fase 1/2: Estadísticos"):
            s, s_c2, n_p, m_min, m_max, my_min, my_max = fut.result()
            suma += s
            suma_c2 += s_c2
            n_pixels_total += n_p
            mins_glob = np.minimum(mins_glob, m_min)
            maxs_glob = np.maximum(maxs_glob, m_max)
            min_y_glob = min(min_y_glob, my_min)
            max_y_glob = max(max_y_glob, my_max)
            
    medias = suma / n_pixels_total
    stds = np.sqrt(suma_c2 / n_pixels_total - medias**2)

    # --- PASO 2: Histogramas en paralelo ---
    BINS = 100
    z_bins = np.linspace(-5, 5, BINS + 1) # Rango fijo seguro para Z-Score
    mm_bins = np.linspace(0, 1, BINS + 1)
    y_bins = np.linspace(min_y_glob, max_y_glob, BINS + 1)
    
    # Acumuladores globales de cuentas
    tot_z_torm = np.zeros((n_vars, BINS))
    tot_z_notorm = np.zeros((n_vars, BINS))
    tot_mm_torm = np.zeros((n_vars, BINS))
    tot_mm_notorm = np.zeros((n_vars, BINS))
    tot_y_torm = np.zeros(BINS)
    tot_y_notorm = np.zeros(BINS)
    
    tareas = [(f, medias, stds, mins_glob, maxs_glob, z_bins, mm_bins, y_bins, n_vars) for f in files]
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futs = [executor.submit(worker_histogramas, t) for t in tareas]
        for fut in tqdm(as_completed(futs), total=len(files), desc="Fase 2/2: Histogramas"):
            c_zt, c_zn, c_mmt, c_mmn, c_yt, c_yn = fut.result()
            tot_z_torm += c_zt
            tot_z_notorm += c_zn
            tot_mm_torm += c_mmt
            tot_mm_notorm += c_mmn
            tot_y_torm += c_yt
            tot_y_notorm += c_yn

    # --- PASO 3: Graficar ---
    print("Generando el gráfico...")
    COLOR_TORM = "#722F37"
    COLOR_NOTORM = "#A0A0A0"
    
    # Grilla dinámica basada en la cantidad de variables
    n_filas = n_vars + 1
    alto_figura = 6 * n_filas # Mantiene la proporción visual
    fig, axes = plt.subplots(n_filas, 2, figsize=(24, alto_figura))
    fig.suptitle("Comparación de Normalizaciones: Z-score vs Min-Max", fontsize=24, fontweight="bold")
    
    axes[0, 0].set_title(f"Z-SCORE\n{var_configs[0]['nombre'].upper()}", fontweight="bold", fontsize=16)
    axes[0, 1].set_title(f"MIN-MAX\n{var_configs[0]['nombre'].upper()}", fontweight="bold", fontsize=16)

    def graficar_canal(ax, bins_edges, counts_notorm, counts_torm, titulo=None):
        ax.hist(bins_edges[:-1], bins=bins_edges, weights=counts_notorm, color=COLOR_NOTORM, alpha=0.8, label="lightdens ≤ 0")
        ax.hist(bins_edges[:-1], bins=bins_edges, weights=counts_torm, color=COLOR_TORM, alpha=0.8, label="lightdens > 0")
        
        todas_counts = np.concatenate([counts_notorm, counts_torm])
        todas_counts = todas_counts[todas_counts > 0]
        if len(todas_counts) > 0:
            ax.set_ylim(top=np.percentile(todas_counts, 95) * 1.1)
            
        if titulo: ax.set_title(titulo, fontweight="bold")
        ax.set_xlabel("Valor normalizado")
        ax.set_ylabel("Frecuencia absoluta")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    for i in range(n_vars):
        nombre = var_configs[i]["nombre"].upper()
        titulo = nombre if i > 0 else None
        graficar_canal(axes[i, 0], z_bins, tot_z_notorm[i], tot_z_torm[i], titulo)
        graficar_canal(axes[i, 1], mm_bins, tot_mm_notorm[i], tot_mm_torm[i], titulo)

    # Gráfico de y (target) ubicado siempre en la última fila dinámicamente
    for col in [0, 1]:
        ax = axes[n_vars, col]
        ax.hist(y_bins[:-1], bins=y_bins, weights=tot_y_notorm, color=COLOR_NOTORM, alpha=0.8, label="lightdens ≤ 0")
        ax.hist(y_bins[:-1], bins=y_bins, weights=tot_y_torm, color=COLOR_TORM, alpha=0.8, label="lightdens > 0")
        
        todas_counts = np.concatenate([tot_y_notorm, tot_y_torm])
        todas_counts = todas_counts[todas_counts > 0]
        if len(todas_counts) > 0: ax.set_ylim(top=np.percentile(todas_counts, 95) * 1.1)
            
        ax.set_title("LIGHTDENS — log(x + 0.1)", fontweight="bold")
        ax.set_xlabel("log(LIGHTDENS + 0.1)")
        ax.set_ylabel("Frecuencia")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    ruta_guardado = "/serafin/gcamilo/Trabajo_Final/images/comparacion_distribuciones.png"
    plt.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    print(f"\n¡Imagen guardada con éxito en '{ruta_guardado}'!")
    plt.show()

# --- EJECUCIÓN ---
plot_comparacion_distribuciones_paralelo(f"{OUTPUT_DIR}/train", VAR_CONFIGS, max_workers=16)

In [ ]:
import numpy as np
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt

# ==========================================
# FASE 1: WORKER DE ESTADÍSTICOS CTT
# ==========================================
def worker_stats_ctt(args):
    filepath, idx_ctt, clip_val = args
    data = np.load(filepath)
    ctt = data["X"][idx_ctt].astype(np.float64)
    ctt_clip = np.minimum(ctt, clip_val)
    
    suma_orig = ctt.sum()
    suma_c2_orig = (ctt**2).sum()
    suma_clip = ctt_clip.sum()
    suma_c2_clip = (ctt_clip**2).sum()
    
    n_pixels = ctt.size
    
    min_orig = ctt.min()
    max_orig = ctt.max()
    min_clip = ctt_clip.min()
    max_clip = ctt_clip.max()
    
    return (suma_orig, suma_c2_orig, suma_clip, suma_c2_clip, n_pixels, 
            min_orig, max_orig, min_clip, max_clip)

# ==========================================
# FASE 2: WORKER DE HISTOGRAMAS CTT
# ==========================================
def worker_histogramas_ctt(args):
    (filepath, idx_ctt, clip_val, 
     media_orig, std_orig, min_orig, max_orig, 
     media_clip, std_clip, min_clip, max_clip, 
     z_bins, mm_bins) = args
     
    data = np.load(filepath)
    ctt = data["X"][idx_ctt].astype(np.float64)
    ctt_clip = np.minimum(ctt, clip_val)
    y = data["y"]
    
    mask_torm = (y.flatten() > 0)
    mask_notorm = ~mask_torm
    
    # Normalizaciones locales
    flat_z_orig = ((ctt - media_orig) / (std_orig + 1e-8)).flatten()
    flat_z_clip = ((ctt_clip - media_clip) / (std_clip + 1e-8)).flatten()
    flat_mm_orig = ((ctt - min_orig) / (max_orig - min_orig + 1e-8)).flatten()
    flat_mm_clip = ((ctt_clip - min_clip) / (max_clip - min_clip + 1e-8)).flatten()
    
    # Histogramas
    c_zo_t, _ = np.histogram(flat_z_orig[mask_torm], bins=z_bins)
    c_zo_n, _ = np.histogram(flat_z_orig[mask_notorm], bins=z_bins)
    
    c_zc_t, _ = np.histogram(flat_z_clip[mask_torm], bins=z_bins)
    c_zc_n, _ = np.histogram(flat_z_clip[mask_notorm], bins=z_bins)
    
    c_mmo_t, _ = np.histogram(flat_mm_orig[mask_torm], bins=mm_bins)
    c_mmo_n, _ = np.histogram(flat_mm_orig[mask_notorm], bins=mm_bins)
    
    c_mmc_t, _ = np.histogram(flat_mm_clip[mask_torm], bins=mm_bins)
    c_mmc_n, _ = np.histogram(flat_mm_clip[mask_notorm], bins=mm_bins)
    
    return c_zo_t, c_zo_n, c_zc_t, c_zc_n, c_mmo_t, c_mmo_n, c_mmc_t, c_mmc_n

# ==========================================
# FUNCIÓN PRINCIPAL ORQUESTADORA
# ==========================================
def plot_ctt_clip_paralelo(split_dir, var_configs, clip_val=0, max_workers=16):
    files = sorted(Path(split_dir).glob("*.npz"))
    idx_ctt = next(i for i, vc in enumerate(var_configs) if vc["nombre"] == "ctt")
    
    # --- PASO 1: Estadísticos en paralelo ---
    suma_orig = suma_clip = suma_c2_orig = suma_c2_clip = 0.0
    min_orig = min_clip = np.inf
    max_orig = max_clip = -np.inf
    n_total = 0
    
    tareas_stats = [(f, idx_ctt, clip_val) for f in files]
    
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futs = [executor.submit(worker_stats_ctt, t) for t in tareas_stats]
        for fut in tqdm(as_completed(futs), total=len(files), desc="Fase 1/2: Estadísticos CTT"):
            s_o, sc2_o, s_c, sc2_c, n_p, m_min_o, m_max_o, m_min_c, m_max_c = fut.result()
            suma_orig += s_o
            suma_c2_orig += sc2_o
            suma_clip += s_c
            suma_c2_clip += sc2_c
            n_total += n_p
            
            min_orig = min(min_orig, m_min_o); max_orig = max(max_orig, m_max_o)
            min_clip = min(min_clip, m_min_c); max_clip = max(max_clip, m_max_c)
            
    media_orig = suma_orig / n_total
    std_orig = np.sqrt(suma_c2_orig / n_total - media_orig**2)
    media_clip = suma_clip / n_total
    std_clip = np.sqrt(suma_c2_clip / n_total - media_clip**2)

    print(f"\nMedia CTT original:  {media_orig:.2f} | Std original: {std_orig:.2f}")
    print(f"Media CTT clipeada:  {media_clip:.2f} | Std clipeada: {std_clip:.2f}\n")

    # --- PASO 2: Histogramas en paralelo ---
    BINS = 100
    z_bins = np.linspace(-5, 5, BINS + 1)
    mm_bins = np.linspace(0, 1, BINS + 1)
    
    tot_zo_t = np.zeros(BINS); tot_zo_n = np.zeros(BINS)
    tot_zc_t = np.zeros(BINS); tot_zc_n = np.zeros(BINS)
    tot_mmo_t = np.zeros(BINS); tot_mmo_n = np.zeros(BINS)
    tot_mmc_t = np.zeros(BINS); tot_mmc_n = np.zeros(BINS)
    
    tareas_hists = [(f, idx_ctt, clip_val, media_orig, std_orig, min_orig, max_orig, 
                     media_clip, std_clip, min_clip, max_clip, z_bins, mm_bins) for f in files]
                     
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futs = [executor.submit(worker_histogramas_ctt, t) for t in tareas_hists]
        for fut in tqdm(as_completed(futs), total=len(files), desc="Fase 2/2: Histogramas CTT"):
            c_zot, c_zon, c_zct, c_zcn, c_mmot, c_mmon, c_mmct, c_mmcn = fut.result()
            tot_zo_t += c_zot; tot_zo_n += c_zon
            tot_zc_t += c_zct; tot_zc_n += c_zcn
            tot_mmo_t += c_mmot; tot_mmo_n += c_mmon
            tot_mmc_t += c_mmct; tot_mmc_n += c_mmcn

    # --- PASO 3: Graficar ---
    print("Generando el gráfico...")
    COLOR_TORM = "#722F37"
    COLOR_NOTORM = "#A0A0A0"
    
    fig, axes = plt.subplots(2, 2, figsize=(24, 14))
    fig.suptitle(f"CTT: Original vs Clipeada (clip_max={clip_val}°C)", fontsize=20, fontweight="bold")
    
    configs = [
        (axes[0, 0], z_bins, tot_zo_n, tot_zo_t, "Z-score | CTT Original"),
        (axes[0, 1], z_bins, tot_zc_n, tot_zc_t, f"Z-score | CTT Clipeada (≤{clip_val}°C)"),
        (axes[1, 0], mm_bins, tot_mmo_n, tot_mmo_t, "Min-Max | CTT Original"),
        (axes[1, 1], mm_bins, tot_mmc_n, tot_mmc_t, f"Min-Max | CTT Clipeada (≤{clip_val}°C)"),
    ]

    for ax, bins_edges, counts_n, counts_t, titulo in configs:
        ax.hist(bins_edges[:-1], bins=bins_edges, weights=counts_n, color=COLOR_NOTORM, alpha=0.8, label="lightdens ≤ 0")
        ax.hist(bins_edges[:-1], bins=bins_edges, weights=counts_t, color=COLOR_TORM, alpha=0.8, label="lightdens > 0")
        
        todas_counts = np.concatenate([counts_n, counts_t])
        todas_counts = todas_counts[todas_counts > 0]
        if len(todas_counts) > 0:
            ax.set_ylim(top=np.percentile(todas_counts, 95) * 1.1)
            
        ax.set_title(titulo, fontweight="bold", fontsize=14)
        ax.set_xlabel("Valor normalizado")
        ax.set_ylabel("Frecuencia absoluta")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    ruta = "/serafin/gcamilo/Trabajo_Final/images/ctt_clip_comparacion.png"
    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    print(f"\n¡Imagen guardada en '{ruta}'!")
    plt.show()

# --- EJECUCIÓN ---
plot_ctt_clip_paralelo(f"{OUTPUT_DIR}/train", VAR_CONFIGS, clip_val=0, max_workers=16)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt

# ==========================================
# FASE 1: WORKER DE PRODUCTOS CRUZADOS
# ==========================================
def worker_correlacion(args):
    filepath, n_vars = args
    data = np.load(filepath)
    X = data["X"].astype(np.float64)
    y = data["y"].astype(np.float64)
    
    # Aplanamos (H*W, Canales) y juntamos todo
    X_flat = X.reshape(n_vars, -1).T
    y_flat = y.flatten().reshape(-1, 1)
    combined = np.hstack([X_flat, y_flat]) # Forma: (N_pixeles, n_vars + 1)
    
    mask_torm = (y_flat > 0).flatten()
    mask_notorm = ~mask_torm
    
    comb_t = combined[mask_torm]
    comb_n = combined[mask_notorm]
    
    # Estadísticos para torm (rayos > 0)
    n_t = comb_t.shape[0]
    sum_t = comb_t.sum(axis=0) if n_t > 0 else np.zeros(n_vars + 1)
    # El producto matricial X.T @ X nos da la suma de productos cruzados de una vez
    prod_t = comb_t.T @ comb_t if n_t > 0 else np.zeros((n_vars + 1, n_vars + 1))
    
    # Estadísticos para notorm (rayos == 0)
    n_n = comb_n.shape[0]
    sum_n = comb_n.sum(axis=0) if n_n > 0 else np.zeros(n_vars + 1)
    prod_n = comb_n.T @ comb_n if n_n > 0 else np.zeros((n_vars + 1, n_vars + 1))
    
    return n_t, sum_t, prod_t, n_n, sum_n, prod_n

# ==========================================
# FUNCIÓN MATEMÁTICA PARA CORRELACIÓN
# ==========================================
def calcular_matriz_correlacion(n, suma, prod):
    """Calcula Pearson a partir de N, Sumas y Productos cruzados."""
    if n == 0:
        return np.zeros_like(prod)
    
    # Covarianza (numerador)
    cov_num = prod - np.outer(suma, suma) / n
    # Varianzas (diagonal)
    var_num = np.diag(cov_num)
    
    # Matriz de desviaciones estándar combinadas (denominador)
    std_matrix = np.sqrt(np.outer(var_num, var_num))
    
    # Evitar divisiones por cero (ej: en notorm, la varianza de y es 0 porque y siempre es 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        corr = cov_num / std_matrix
    
    return np.nan_to_num(corr, nan=0.0)

# ==========================================
# FUNCIÓN PRINCIPAL ORQUESTADORA
# ==========================================
def plot_heatmap_correlacion_paralelo(split_dir, var_configs, max_workers=16):
    files = sorted(Path(split_dir).glob("*.npz"))
    n_vars = len(var_configs)
    k = n_vars + 1 # Variables de entrada + TARGET
    
    # Acumuladores globales
    tot_n_t = 0; tot_sum_t = np.zeros(k); tot_prod_t = np.zeros((k, k))
    tot_n_n = 0; tot_sum_n = np.zeros(k); tot_prod_n = np.zeros((k, k))
    
    tareas = [(f, n_vars) for f in files]
    
    # --- PASO 1: Recolección paralela ---
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futs = [executor.submit(worker_correlacion, t) for t in tareas]
        for fut in tqdm(as_completed(futs), total=len(files), desc="Calculando covarianzas"):
            n_t, s_t, p_t, n_n, s_n, p_n = fut.result()
            
            tot_n_t += n_t; tot_sum_t += s_t; tot_prod_t += p_t
            tot_n_n += n_n; tot_sum_n += s_n; tot_prod_n += p_n

    # --- PASO 2: Calcular correlaciones finales ---
    # Global (Torm + Notorm)
    tot_n_full = tot_n_t + tot_n_n
    tot_sum_full = tot_sum_t + tot_sum_n
    tot_prod_full = tot_prod_t + tot_prod_n
    
    matriz_corr_torm = calcular_matriz_correlacion(tot_n_t, tot_sum_t, tot_prod_t)
    matriz_corr_notorm = calcular_matriz_correlacion(tot_n_n, tot_sum_n, tot_prod_n)
    matriz_corr_full = calcular_matriz_correlacion(tot_n_full, tot_sum_full, tot_prod_full)
    
    # Convertimos a DataFrame para usar con Seaborn
    columnas = [v["nombre"].upper() for v in var_configs] + ["LIGHTDENS"]
    df_corr_torm = pd.DataFrame(matriz_corr_torm, index=columnas, columns=columnas)
    df_corr_notorm = pd.DataFrame(matriz_corr_notorm, index=columnas, columns=columnas)
    df_corr_full = pd.DataFrame(matriz_corr_full, index=columnas, columns=columnas)

    # --- PASO 3: Graficar ---
    print("Generando los heatmaps...")
    fig, axes = plt.subplots(1, 3, figsize=(28, 9))
    fig.suptitle("Análisis de Correlación", fontsize=20, fontweight="bold")

    cbar_ax = fig.add_axes([0.25, 0.06, 0.50, 0.03]) 

    # Mapa 1: Con Actividad Eléctrica
    sns.heatmap(df_corr_torm, annot=True, cmap='RdBu_r', center=0, fmt=".2f", 
                ax=axes[0], cbar=False, vmin=-1, vmax=1)
    axes[0].set_title("1. SOLO PUNTOS CON RAYOS\n(Dinámica de la Descarga)", fontsize=14)

    # Mapa 2: Sin Actividad Eléctrica
    sns.heatmap(df_corr_notorm, annot=True, cmap='RdBu_r', center=0, fmt=".2f", 
                ax=axes[1], cbar=False, vmin=-1, vmax=1)
    axes[1].set_title("2. SOLO PUNTOS SIN RAYOS\n(Estado de Reposo)", fontsize=14)

    # Mapa 3: Todo junto
    sns.heatmap(df_corr_full, annot=True, cmap='RdBu_r', center=0, fmt=".2f", 
                ax=axes[2], vmin=-1, vmax=1,
                cbar=True, cbar_ax=cbar_ax, cbar_kws={'orientation': 'horizontal'})
    axes[2].set_title("3. TODO JUNTO\n(Comportamiento General)", fontsize=14)

    plt.tight_layout(rect=[0, 0.12, 1, 0.95])
    ruta = "/serafin/gcamilo/Trabajo_Final/images/heatmap_correlacion.png"
    plt.savefig(ruta, dpi=150)
    print(f"\n¡Imagen guardada en '{ruta}'!")
    plt.show()

# --- EJECUCIÓN ---
plot_heatmap_correlacion_paralelo(f"{OUTPUT_DIR}/train", VAR_CONFIGS, max_workers=16)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib.pyplot as plt

# ==========================================
# FUNCIÓN MATEMÁTICA PARA CORRELACIÓN
# ==========================================
def calcular_matriz_correlacion(n, suma, prod):
    """Calcula Pearson a partir de N, Sumas y Productos cruzados."""
    if n == 0:
        return np.zeros_like(prod)
    
    # Covarianza (numerador)
    cov_num = prod - np.outer(suma, suma) / n
    # Varianzas (diagonal)
    var_num = np.diag(cov_num)
    
    # Matriz de desviaciones estándar combinadas (denominador)
    std_matrix = np.sqrt(np.outer(var_num, var_num))
    
    # Evitar divisiones por cero 
    with np.errstate(divide='ignore', invalid='ignore'):
        corr = cov_num / std_matrix
    
    return np.nan_to_num(corr, nan=0.0)

# ==========================================
# FASE 1: WORKER DE PRODUCTOS CRUZADOS CON CLIPPING
# ==========================================
def worker_correlacion_ctt_clip(args):
    filepath, n_vars, ctt_idx = args
    data = np.load(filepath)
    X = data["X"].astype(np.float64)
    y = data["y"].astype(np.float64)
    
    # CLIPPING: Todos los valores positivos a 0
    if ctt_idx != -1:
        X[ctt_idx] = np.minimum(X[ctt_idx], 0)
    
    # Aplanamos (H*W, Canales) y juntamos todo
    X_flat = X.reshape(n_vars, -1).T
    y_flat = y.flatten().reshape(-1, 1)
    combined = np.hstack([X_flat, y_flat]) # Forma: (N_pixeles, n_vars + 1)
    
    mask_torm = (y_flat > 0).flatten()
    mask_notorm = ~mask_torm
    
    comb_t = combined[mask_torm]
    comb_n = combined[mask_notorm]
    
    # Estadísticos para torm (rayos > 0)
    n_t = comb_t.shape[0]
    sum_t = comb_t.sum(axis=0) if n_t > 0 else np.zeros(n_vars + 1)
    prod_t = comb_t.T @ comb_t if n_t > 0 else np.zeros((n_vars + 1, n_vars + 1))
    
    # Estadísticos para notorm (rayos == 0)
    n_n = comb_n.shape[0]
    sum_n = comb_n.sum(axis=0) if n_n > 0 else np.zeros(n_vars + 1)
    prod_n = comb_n.T @ comb_n if n_n > 0 else np.zeros((n_vars + 1, n_vars + 1))
    
    return n_t, sum_t, prod_t, n_n, sum_n, prod_n

# ==========================================
# FUNCIÓN PRINCIPAL ORQUESTADORA
# ==========================================
def plot_heatmap_correlacion_ctt_clip_paralelo(split_dir, var_configs, max_workers=16):
    files = sorted(Path(split_dir).glob("*.npz"))
    n_vars = len(var_configs)
    k = n_vars + 1 # Variables de entrada + TARGET
    
    columnas = [v["nombre"].upper() for v in var_configs] + ["LIGHTDENS"]
    
    # 1. Buscamos en qué posición está la variable CTT
    try:
        ctt_idx = columnas.index("CTT")
    except ValueError:
        ctt_idx = -1
        print("Advertencia: No se encontró la variable CTT en VAR_CONFIGS.")
        
    # Acumuladores globales
    tot_n_t = 0; tot_sum_t = np.zeros(k); tot_prod_t = np.zeros((k, k))
    tot_n_n = 0; tot_sum_n = np.zeros(k); tot_prod_n = np.zeros((k, k))
    
    tareas = [(f, n_vars, ctt_idx) for f in files]
    
    # --- PASO 1: Recolección paralela ---
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futs = [executor.submit(worker_correlacion_ctt_clip, t) for t in tareas]
        for fut in tqdm(as_completed(futs), total=len(files), desc="Calculando covarianzas (CTT <= 0)"):
            n_t, s_t, p_t, n_n, s_n, p_n = fut.result()
            
            tot_n_t += n_t; tot_sum_t += s_t; tot_prod_t += p_t
            tot_n_n += n_n; tot_sum_n += s_n; tot_prod_n += p_n

    # --- PASO 2: Calcular correlaciones finales ---
    # Global (Torm + Notorm)
    tot_n_full = tot_n_t + tot_n_n
    tot_sum_full = tot_sum_t + tot_sum_n
    tot_prod_full = tot_prod_t + tot_prod_n
    
    matriz_corr_torm = calcular_matriz_correlacion(tot_n_t, tot_sum_t, tot_prod_t)
    matriz_corr_notorm = calcular_matriz_correlacion(tot_n_n, tot_sum_n, tot_prod_n)
    matriz_corr_full = calcular_matriz_correlacion(tot_n_full, tot_sum_full, tot_prod_full)
    
    # Convertimos a DataFrame para usar con Seaborn
    df_corr_torm = pd.DataFrame(matriz_corr_torm, index=columnas, columns=columnas)
    df_corr_notorm = pd.DataFrame(matriz_corr_notorm, index=columnas, columns=columnas)
    df_corr_full = pd.DataFrame(matriz_corr_full, index=columnas, columns=columnas)

    # --- PASO 3: Graficar ---
    print("Generando los heatmaps...")
    fig, axes = plt.subplots(1, 3, figsize=(28, 9))
    fig.suptitle("Análisis de Correlación (CTT positivos a 0)", fontsize=20, fontweight="bold")

    cbar_ax = fig.add_axes([0.25, 0.06, 0.50, 0.03]) 

    # Mapa 1: Con Actividad Eléctrica
    sns.heatmap(df_corr_torm, annot=True, cmap='RdBu_r', center=0, fmt=".2f", 
                ax=axes[0], cbar=False, vmin=-1, vmax=1)
    axes[0].set_title("1. SOLO PUNTOS CON RAYOS\n(Dinámica de la Descarga)", fontsize=14)

    # Mapa 2: Sin Actividad Eléctrica
    sns.heatmap(df_corr_notorm, annot=True, cmap='RdBu_r', center=0, fmt=".2f", 
                ax=axes[1], cbar=False, vmin=-1, vmax=1)
    axes[1].set_title("2. SOLO PUNTOS SIN RAYOS\n(Estado de Reposo)", fontsize=14)

    # Mapa 3: Todo junto
    sns.heatmap(df_corr_full, annot=True, cmap='RdBu_r', center=0, fmt=".2f", 
                ax=axes[2], vmin=-1, vmax=1,
                cbar=True, cbar_ax=cbar_ax, cbar_kws={'orientation': 'horizontal'})
    axes[2].set_title("3. TODO JUNTO\n(Comportamiento General)", fontsize=14)

    plt.tight_layout(rect=[0, 0.12, 1, 0.95])
    ruta = "/serafin/gcamilo/Trabajo_Final/images/heatmap_correlacion_ctt_cero.png"
    plt.savefig(ruta, dpi=150)
    print(f"\n¡Imagen guardada en '{ruta}'!")
    plt.show()

# --- EJECUCIÓN ---
plot_heatmap_correlacion_ctt_clip_paralelo(f"{OUTPUT_DIR}/train", VAR_CONFIGS, max_workers=16)

In [ ]:
%%writefile ../modulos/tormentaDataset.py
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class TormentaDataset(Dataset):
    def __init__(self, split_dir, stats_path):
        self.files = sorted(Path(split_dir).glob("*.npz"))
        stats = np.load(stats_path)
        # Shape (3,1,1) para que el broadcast funcione sobre (3,297,297)
        self.min_X = torch.tensor(stats["min_X"], dtype=torch.float32).view(-1, 1, 1)
        self.max_X = torch.tensor(stats["max_X"], dtype=torch.float32).view(-1, 1, 1)
        self.min_y = float(stats["min_y"])
        self.max_y = float(stats["max_y"])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        X = torch.tensor(data["X"], dtype=torch.float32)
        y = torch.tensor(data["y"], dtype=torch.float32).unsqueeze(0)

        # Normalización Min-Max al rango [0, 1] usando los extremos globales
        X = (X - self.min_X) / (self.max_X - self.min_X + 1e-8)  # 1e-8 evita división por cero

        # Normalización logarítmica
        y_log = torch.log(y + 0.1)

        # [0,1] para la sigmoide (SOLO si se va a usar FACL como loss)
        y = (y_log - self.min_y) / (self.max_y - self.min_y + 1e-8)

        return X, y

    def desnormalizar_y(self, y_norm):
        """Inversa completa: [0,1] → log(y+0.1) → LIGHTDENS real."""
        y_log = y_norm * (self.max_y - self.min_y) + self.min_y
        return torch.clamp(torch.exp(y_log) - 0.1, min=0)

    def __repr__(self):
        return f'Dataset con {len(self.files)} archivos'


In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
from modulos.tormentaDataset import TormentaDataset
from torch.utils.data import DataLoader
stats_path = f"{OUTPUT_DIR}/stats.npz"
train_ds = TormentaDataset(f"{OUTPUT_DIR}/train", stats_path)
val_ds   = TormentaDataset(f"{OUTPUT_DIR}/val",   stats_path)

train_dl = DataLoader(train_ds, batch_size=4, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=4, shuffle=False)

In [ ]:
# 1. Chequeamos las longitudes
print("--- TAMAÑOS ---")
print(f"Muestras en Train Dataset: {len(train_ds)}")
print(f"Batches (Lotes) en Train DataLoader: {len(train_dl)}")

# 2. Extraemos UN batch
X_batch, y_batch = next(iter(train_dl))

print("\n--- SHAPES Y TIPOS ---")
print(f"Shape de X: {X_batch.shape} -> (Batch, Canales, Alto, Ancho)")
print(f"Shape de y: {y_batch.shape} -> (Batch, Canales, Alto, Ancho)")
print(f"Tipo de X: {X_batch.dtype}")
print(f"Tipo de y: {y_batch.dtype}")

# 3. Chequeamos que no haya valores rotos (NaNs)
print("\n--- VALORES ---")
print(f"X min/max: {X_batch.min().item():.2f} / {X_batch.max().item():.2f}")
print(f"y min/max: {y_batch.min().item():.2f} / {y_batch.max().item():.2f}")

In [ ]:
# Agarramos la primera muestra del batch de 4
X_muestra = X_batch[0]  # Shape: (3, 297, 297)
y_muestra = y_batch[0]  # Shape: (1, 297, 297)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot Canal 0 de X (ej. Reflectividad)
im0 = axes[0].imshow(X_muestra[0].numpy(), cmap="jet")
axes[0].set_title("Feature X (Canal 0)")
fig.colorbar(im0, ax=axes[0])

# Plot Target y (Densidad de Rayos)
im1 = axes[1].imshow(y_muestra[0].numpy(), cmap="magma")
axes[1].set_title("Target y (LIGHTDENS)")
fig.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# Fase 5: Dataset temporal (t, Δ1, Δ2)

Dataset **aparte** (`dataset_temporal/`), el original (`dataset/`) no se toca. Mismas muestras, mismo split train/val, mismo `y`; solo cambia `X`, que pasa de 3 a 9 canales:

| Canales | Contenido |
|---|---|
| 0–2 | `MDBZ_t`, `HIELO_t`, `CTT_t` (variables en el instante t) |
| 3–5 | `MDBZ_d1`, `HIELO_d1`, `CTT_d1` = X(t) − X(t−1) |
| 6–8 | `MDBZ_d2`, `HIELO_d2`, `CTT_d2` = X(t−1) − X(t−2) |

Las restas se hacen sobre los valores **crudos** (sin normalizar) y después se normaliza cada canal con su propio min-max global de train.

In [ ]:
# ==========================================================
# Configuración del dataset temporal
# ==========================================================
OUTPUT_DIR_T = "/serafin/gcamilo/Trabajo_Final/dataset_temporal"
assert os.path.abspath(OUTPUT_DIR_T) != os.path.abspath(OUTPUT_DIR), "¡OUTPUT_DIR_T apunta al dataset original!"

UMBRAL_LIGHTDENS = 150
H = W = 299 - 2 * CROP         # 297
N_LAGS = 2                      # t-1 y t-2
REUSAR_DATASET_ORIGINAL = True  # si el .npz de un timestep ya existe en dataset/, se reusa su X en vez de recalcularlo desde el wrfout

NOMBRES_VARS = [vc["nombre"].upper() for vc in VAR_CONFIGS]            # ['MDBZ', 'HIELO', 'CTT']
NOMBRES_CANALES = ([f"{n}_t"  for n in NOMBRES_VARS] +                  # t
                   [f"{n}_d1" for n in NOMBRES_VARS] +                  # t - (t-1)
                   [f"{n}_d2" for n in NOMBRES_VARS])                   # (t-1) - (t-2)
N_CANALES_T = len(NOMBRES_CANALES)                                      # 9
print(N_CANALES_T, NOMBRES_CANALES)

In [ ]:
# ==========================================================
# Muestras: cada timestep t con sus archivos t-1 y t-2 (de la MISMA corrida continua)
# ==========================================================
val_ids = set(val_tormentas)

muestras, descartadas = [], []     # muestras: (archivo_t, archivo_t1, archivo_t2, split)
for t in tormentas:
    split = "val" if t["id"] in val_ids else "train"
    lista = eventos[t["corrida"]]
    for idx in t["indices"]:
        if idx < N_LAGS:           # el timestep está al inicio de la corrida: no existe t-2
            descartadas.append((split, lista[idx]))
            continue
        muestras.append((lista[idx], lista[idx - 1], lista[idx - 2], split))

# Los lags tienen que ser exactamente `paso` hacia atrás (las corridas ya son consecutivas por construcción)
for f, f1, f2, _ in muestras:
    assert t_archivo(f) - t_archivo(f1) == paso and t_archivo(f1) - t_archivo(f2) == paso, f"Lags no consecutivos en {f}"

n_total  = len(muestras) + len(descartadas)
print(f"Samples del dataset original: {n_total}")
print(f"Samples del dataset temporal: {len(muestras)} (train {sum(m[3]=='train' for m in muestras)} | val {sum(m[3]=='val' for m in muestras)})")
print(f"Descartados por no tener t-1/t-2 en la corrida: {len(descartadas)}")
for split, f in descartadas:
    print(f"   [{split}] {os.path.basename(f)}")

# Control de fuga entre splits: ¿algún input usa un timestep (t-1 o t-2) que pertenece al OTRO split?
split_de = {m[0]: m[3] for m in muestras} | {f: s for s, f in descartadas}
cruzados = [m for m in muestras if any(split_de.get(f, m[3]) != m[3] for f in (m[1], m[2]))]
print(f"\nSamples cuyo input incluye timesteps del otro split: {len(cruzados)}")
if cruzados:
    print(f"   (con paso = {paso} y GAP_MIN_MIN = {GAP_MIN_MIN} min; revisar si es aceptable)")

In [ ]:
def canales_crudos(filepath):
    """X crudo (N_CANALES_POR_TS, H, W) de un timestep. Reusa el .npz del dataset original si existe."""
    if REUSAR_DATASET_ORIGINAL:
        nombre = os.path.basename(filepath).replace(":", "-") + ".npz"
        for split in ("train", "val"):
            p = os.path.join(OUTPUT_DIR, split, nombre)
            if os.path.exists(p):
                X = np.load(p)["X"]
                if X.shape[0] == N_CANALES_POR_TS:
                    return X.astype(np.float32)
    return extraer_timestep(filepath)          # fallback: calcular desde el wrfout (lento)

def procesar_archivo_temporal(filepath, fp_t1, fp_t2, split, output_dir):
    # y: idéntico al dataset original
    ds = Dataset(filepath)
    y = to_np(ds["LIGHTDENS"][0])[CROP:-CROP, CROP:-CROP].astype(np.float32)
    ds.close()
    y = np.minimum(y, UMBRAL_LIGHTDENS)

    # X: [X(t), X(t) - X(t-1), X(t-1) - X(t-2)]  (restas sobre valores crudos)
    X_t, X_t1, X_t2 = canales_crudos(filepath), canales_crudos(fp_t1), canales_crudos(fp_t2)
    X = np.concatenate([X_t, X_t - X_t1, X_t1 - X_t2], axis=0).astype(np.float32)
    assert X.shape[0] == N_CANALES_T and np.isfinite(X).all(), f"X inválido en {filepath}"

    nombre = os.path.basename(filepath).replace(":", "-")      # mismo nombre que en el dataset original
    np.savez_compressed(os.path.join(output_dir, split, f"{nombre}.npz"), X=X, y=y)
    return split

In [ ]:
# Limpieza SOLO del dataset temporal
for split in ["train", "val"]:
    split_dir = os.path.join(OUTPUT_DIR_T, split)
    if os.path.exists(split_dir):
        shutil.rmtree(split_dir)
    os.makedirs(split_dir, exist_ok=True)

n_train_t = n_val_t = 0
print(f"Lanzando procesamiento en paralelo para {len(muestras)} muestras...")
with ProcessPoolExecutor(max_workers=16) as executor:
    futuros = [executor.submit(procesar_archivo_temporal, f, f1, f2, split, OUTPUT_DIR_T)
               for f, f1, f2, split in muestras]
    for futuro in tqdm(as_completed(futuros), total=len(futuros), desc="Procesando"):
        if futuro.result() == "train":
            n_train_t += 1
        else:
            n_val_t += 1

print(f"\nDataset temporal guardado en {OUTPUT_DIR_T}")
print(f"  Train: {n_train_t} | Val: {n_val_t}")
print(f"Shape X: ({N_CANALES_T}, {H}, {W}) | Shape y: ({H}, {W})")

In [ ]:
# ==========================================================
# Estadísticos globales (SOLO TRAIN) — min/max por canal para la normalización min-max
# ==========================================================
def stats_locales_t(filepath):
    d = np.load(filepath)
    X = d["X"]
    y_log = np.log(d["y"].astype(np.float64) + 0.1)
    return X.min(axis=(1, 2)), X.max(axis=(1, 2)), y_log.min(), y_log.max()

archivos_train_t = sorted(Path(f"{OUTPUT_DIR_T}/train").glob("*.npz"))
min_X_t = np.full(N_CANALES_T,  np.inf, dtype=np.float64)
max_X_t = np.full(N_CANALES_T, -np.inf, dtype=np.float64)
min_y_t, max_y_t = float("inf"), float("-inf")

with ProcessPoolExecutor(max_workers=16) as executor:
    futuros = [executor.submit(stats_locales_t, f) for f in archivos_train_t]
    for futuro in tqdm(as_completed(futuros), total=len(futuros), desc="Estadísticos"):
        mn, mx, mny, mxy = futuro.result()
        min_X_t, max_X_t = np.minimum(min_X_t, mn), np.maximum(max_X_t, mx)
        min_y_t, max_y_t = min(min_y_t, mny), max(max_y_t, mxy)

print("--- ESTADÍSTICOS DE ENTRADA (X) [MIN-MAX] ---")
for i, nombre in enumerate(NOMBRES_CANALES):
    print(f"Canal {i} | {nombre:<9}: min {min_X_t[i]:>12.4f} | max {max_X_t[i]:>12.4f}")
print(f"\nlog(LIGHTDENS+0.1) — min: {min_y_t:.4f}, max: {max_y_t:.4f}")

stats_path_t = f"{OUTPUT_DIR_T}/stats.npz"
np.savez(stats_path_t, min_X=min_X_t, max_X=max_X_t, min_y=min_y_t, max_y=max_y_t,
         canales=np.array(NOMBRES_CANALES))
print(f"\nGuardado en {stats_path_t}")

# Sanity check contra el dataset original: los 3 primeros canales (t) y y deberían coincidir
# (pueden diferir apenas si algún sample descartado tenía el extremo global)
if os.path.exists(f"{OUTPUT_DIR}/stats.npz"):
    s0 = np.load(f"{OUTPUT_DIR}/stats.npz")
    print("\nComparación con dataset original (canales en t):")
    print("  |Δ min_X|:", np.abs(s0["min_X"] - min_X_t[:N_CANALES_POR_TS]))
    print("  |Δ max_X|:", np.abs(s0["max_X"] - max_X_t[:N_CANALES_POR_TS]))
    print(f"  |Δ min_y|: {abs(float(s0['min_y']) - min_y_t):.6f} | |Δ max_y|: {abs(float(s0['max_y']) - max_y_t):.6f}")

In [ ]:
# ==========================================================
# Verificación de integridad + visualización
# ==========================================================
rng = np.random.default_rng(0)
for k in rng.choice(len(muestras), size=5, replace=False):
    f, f1, f2, split = muestras[k]
    nombre = os.path.basename(f).replace(":", "-") + ".npz"
    X_new = np.load(f"{OUTPUT_DIR_T}/{split}/{nombre}")["X"]
    X_t, X_t1, X_t2 = canales_crudos(f), canales_crudos(f1), canales_crudos(f2)
    assert np.allclose(X_new[:3], X_t),            f"canales en t incorrectos: {nombre}"
    assert np.allclose(X_new[3:6], X_t - X_t1),    f"d1 incorrecto: {nombre}"
    assert np.allclose(X_new[6:9], X_t1 - X_t2),   f"d2 incorrecto: {nombre}"
print("✔ Integridad OK (canales t, d1 y d2 coinciden con los timesteps crudos)")

import sys
sys.path.append(os.path.abspath('..'))
from torch.utils.data import DataLoader
from modulos.tormentaDataset import TormentaDataset     # la clase existente ya sirve: min_X/max_X se broadcastean con view(-1,1,1)
train_ds_t = TormentaDataset(f"{OUTPUT_DIR_T}/train", f"{OUTPUT_DIR_T}/stats.npz")
val_ds_t   = TormentaDataset(f"{OUTPUT_DIR_T}/val",   f"{OUTPUT_DIR_T}/stats.npz")
print(train_ds_t, "|", val_ds_t)

X_b, y_b = next(iter(DataLoader(train_ds_t, batch_size=4, shuffle=True)))
print(f"Shape de X: {tuple(X_b.shape)} (esperado (4, {N_CANALES_T}, {H}, {W})) | y: {tuple(y_b.shape)}")
print(f"X min/max por canal: {[f'{a:.2f}/{b:.2f}' for a, b in zip(X_b.amin(dim=(0, 2, 3)).tolist(), X_b.amax(dim=(0, 2, 3)).tolist())]}")

# Un sample, los 9 canales (valores crudos): fila 0 = t, fila 1 = d1, fila 2 = d2
f, f1, f2, split = muestras[int(rng.integers(len(muestras)))]
X_plot = np.load(f"{OUTPUT_DIR_T}/{split}/{os.path.basename(f).replace(':', '-')}.npz")["X"]
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for c, ax in enumerate(axes.ravel()):
    if c < 3:
        im = ax.imshow(X_plot[c], cmap="jet")
    else:
        vmax = max(np.abs(X_plot[c]).max(), 1e-8)
        im = ax.imshow(X_plot[c], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(NOMBRES_CANALES[c]); ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f"{os.path.basename(f)} [{split}]")
plt.tight_layout(); plt.show()

In [ ]:
# -*- coding: utf-8 -*-
"""
Extracción de datos de los wrfout y del dataset para completar los
pendientes en rojo de las Secciones 3.1 y 3.5.4 de la monografía.

Qué responde cada función
-------------------------
estructura_simulaciones(eventos)
    * Cuántas simulaciones distintas hay dentro de los 17 eventos
      (agrupando por el atributo SIMULATION_START_DATE de cada
      wrfout), con fecha de inicio, de fin y horas de cada una.
    * Una estimación del tiempo de reloj de cada simulación a partir
      de la fecha de modificación de los archivos (último menos
      primero). Solo vale si los wrfout no se copiaron después con
      otra fecha: si todos los archivos tienen casi la misma fecha,
      ignorala.
    * Si algún atributo global (microfísica, paso de tiempo, etc.)
      cambia de un archivo a otro, o si la configuración es idéntica
      en todos (salvo las fechas de inicio).
variables_electricas(ruta)
    * Descripción y unidades que el propio modelo guarda para
      LIGHTDENS, LIGHT, INDUC, NONINDUC, las cargas por
      hidrometeoro, etc. Debería resolver qué mide LIGHTDENS.
serie_lightdens(eventos)
    * Para cada salida: suma de LIGHTDENS, de LIGHT (orígenes de
      descarga) y de LIGHTDIS en todo el dominio (la serie que se
      compara con el LMA, Fig. 3.5), máximos, cantidad de píxeles
      activos y de píxeles con más de 150, y si los valores son
      enteros. Guarda serie_electrica.csv (se puede reanudar).
fraccion_sobre_150(dir_dataset)
    * Qué fracción de los píxeles activos del dataset supera 150
      (y percentiles de los valores activos).

Todas las funciones lentas usan n_proc procesos en paralelo
(N_PROC = 32 por defecto; n_proc=1 las corre sin paralelismo).

Uso en JupyterHub (Datasets.ipynb), una celda por paso
-------------------------------------------------------
    import sys; sys.path.append('../figuras')   # o copiar el archivo
    from extraer_metadatos_wrf import *

    # `eventos` ya existe si corriste la celda "Corridas continuas"
    # de Datasets.ipynb; si no:
    # eventos = armar_eventos()

    res_sim = estructura_simulaciones(eventos)     # unos minutos
    variables_electricas(eventos[sorted(eventos)[0]][0])
    serie = serie_lightdens(eventos)               # unos minutos
    fraccion_sobre_150('../dataset_temporal')

Después copiá y pegá en el chat lo que imprimen (o pasame los CSV
simulaciones_wrf.csv y serie_lightdens.csv).
"""
import glob
import json
import multiprocessing as mp
import os
import re
from datetime import datetime
from functools import partial

import numpy as np
import pandas as pd
from netCDF4 import Dataset

try:
    from tqdm.auto import tqdm
except ImportError:                      # sin tqdm, sin barra
    def tqdm(x, **kw):
        return x

PAT = re.compile(
    r"wrfout_d01_(\d{4}-\d{2}-\d{2})_(\d{2})[:\-](\d{2})[:\-](\d{2})")

# Atributos que cambian de una simulación a otra por construcción
# (todos describen el inicio de la simulación).
IGNORAR = {"START_DATE", "SIMULATION_START_DATE", "JULYR", "JULDAY",
           "GMT"}

ELEC_VARS = ["LIGHTDENS", "LIGHT", "LIGHTDIS", "INDUC", "NONINDUC",
             "ELECMAG", "ELECX", "ELECY", "ELECZ", "PC", "PCB",
             "SCW", "SCR", "SCI", "SCS", "SCH", "SCHL", "SCIONA",
             "SCTOT", "RSCGHIS_2D"]


N_PROC = 32      # procesos en paralelo (1 = sin paralelismo)


def _mapear(fn, items, n_proc, desc, pool=None):
    """fn sobre cada item, en orden, con barra de progreso por
    archivo. n_proc > 1 usa procesos (fork), lo que ayuda mucho
    cuando el cuello de botella es leer del disco de red."""
    items = list(items)
    if n_proc <= 1 or len(items) < 2:
        return [fn(x) for x in tqdm(items, desc=desc)]
    cm = None
    if pool is None:
        cm = pool = mp.get_context("fork").Pool(n_proc)
    try:
        return list(tqdm(pool.imap(fn, items, chunksize=4),
                         total=len(items), desc=desc))
    finally:
        if cm is not None:
            cm.close()
            cm.join()


def t_nombre(ruta):
    """Fecha y hora de la salida, tomadas del nombre del archivo."""
    m = PAT.search(os.path.basename(str(ruta)))
    return datetime.strptime(
        f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}",
        "%Y-%m-%d %H:%M:%S")


def armar_eventos(dirs=("~/jupyter/salidasWRF2018",
                        "../salidasWRF2019")):
    """Misma lógica que la celda de Datasets.ipynb: agrupa los
    wrfout en tramos de salidas consecutivas (un evento = fecha de
    inicio -> lista ordenada de archivos)."""
    archivos = []
    for d in dirs:
        archivos += glob.glob(os.path.join(os.path.expanduser(d),
                                           "wrfout_d01_*"))
    archivos.sort(key=t_nombre)
    ts = [t_nombre(f) for f in archivos]
    paso = min(b - a for a, b in zip(ts[:-1], ts[1:]))
    eventos, actual = {}, [archivos[0]]
    for f1, t0, t1 in zip(archivos[1:], ts[:-1], ts[1:]):
        if t1 - t0 == paso:
            actual.append(f1)
        else:
            eventos[f"{t_nombre(actual[0]):%Y-%m-%d}"] = actual
            actual = [f1]
    eventos[f"{t_nombre(actual[0]):%Y-%m-%d}"] = actual
    print(f"{sum(map(len, eventos.values()))} archivos, "
          f"{len(eventos)} eventos, paso {paso}")
    return eventos


# ---------------------------------------------------------------
# 1. Estructura de las simulaciones y consistencia de la config.
# ---------------------------------------------------------------
def _attrs(ruta):
    with Dataset(ruta) as ds:
        return {k: str(ds.getncattr(k)) for k in ds.ncattrs()}


def _w_estructura(f, ref):
    a = _attrs(f)
    dif = tuple((k, ref.get(k), a.get(k)) for k in set(a) | set(ref)
                if k not in IGNORAR and a.get(k) != ref.get(k))
    return (a.get("SIMULATION_START_DATE"), os.path.getmtime(f), dif)


def estructura_simulaciones(eventos, ruta_csv="simulaciones_wrf.csv",
                            cada=1, n_proc=N_PROC):
    """Agrupa las salidas por SIMULATION_START_DATE dentro de cada
    evento y compara los atributos globales de todos los archivos
    con los del primero. `cada=k` lee un archivo de cada k (más
    rápido, pero las fronteras entre simulaciones quedan con
    precisión de k salidas); siempre se leen el primero y el último
    de cada evento."""
    sel = []
    for ev in sorted(eventos):
        lista = eventos[ev]
        sel += [(ev, f) for i, f in enumerate(lista)
                if cada <= 1 or i % cada == 0 or i == len(lista) - 1]
    ref = _attrs(sel[0][1])
    with open("atributos_globales_ref.json", "w",
              encoding="utf8") as fh:
        json.dump({"archivo": os.path.basename(sel[0][1]),
                   "atributos": ref}, fh, indent=1,
                  ensure_ascii=False)
    out = _mapear(partial(_w_estructura, ref=ref),
                  [f for _, f in sel], n_proc, "Archivos")
    difieren, filas = {}, []
    for (ev, f), (sim, mt, dif) in zip(sel, out):
        for clave in dif:
            d = difieren.setdefault(clave,
                                    [0, os.path.basename(f)])
            d[0] += 1
        filas.append((ev, t_nombre(f), sim, mt))
    df = pd.DataFrame(filas, columns=["evento", "t", "sim_inicio",
                                      "mtime"])
    res = (df.groupby(["evento", "sim_inicio"], sort=True)
             .agg(salidas_leidas=("t", "size"), primera=("t", "min"),
                  ultima=("t", "max"), m_ini=("mtime", "min"),
                  m_fin=("mtime", "max")).reset_index())
    res["horas"] = ((res.ultima - res.primera)
                    .dt.total_seconds() / 3600).round(2)
    # estimación del reloj: ver el docstring del módulo
    res["reloj_h"] = ((res.m_fin - res.m_ini) / 3600).round(2)
    res["reloj_h_por_dia_sim"] = (res.reloj_h
                                  / (res.horas / 24)).round(2)
    res = res.drop(columns=["m_ini", "m_fin"])
    res.to_csv(ruta_csv, index=False)
    print(res.to_string(index=False))
    print(f"\nSimulaciones distintas (por SIMULATION_START_DATE): "
          f"{res.sim_inicio.nunique()} en {res.evento.nunique()} "
          f"eventos. Guardado: {ruta_csv}")
    if difieren:
        print("\nAtributos globales que CAMBIAN respecto del primer "
              "archivo (sin contar fechas de inicio):")
        for (k, v0, v1), (n, ej) in difieren.items():
            print(f"  {k}: {v0} -> {v1}  ({n} archivos, p. ej. {ej})")
    else:
        print(f"\nTodos los atributos globales son idénticos al del "
              f"primer archivo (salvo las fechas de inicio) en los "
              f"{len(df)} archivos leídos.")
    return res


# ---------------------------------------------------------------
# 2. Metadatos de las variables eléctricas
# ---------------------------------------------------------------
def variables_electricas(ruta, nombres=ELEC_VARS):
    """Imprime dimensiones, unidades y descripción que el modelo
    guarda para cada variable eléctrica de un wrfout."""
    with Dataset(ruta) as ds:
        print("Archivo:", os.path.basename(str(ruta)))
        for n in nombres:
            if n not in ds.variables:
                print(f"{n:11s} -- no está en el archivo")
                continue
            v = ds.variables[n]
            print(f"{n:11s} dims={v.dimensions} "
                  f"unidades={getattr(v, 'units', '?')!r} "
                  f"descripción={getattr(v, 'description', '?')!r}")
        print("\nOtras variables con 'ELEC', 'LIGHT' o 'CHARGE' en "
              "el nombre:",
              [k for k in ds.variables
               if re.search("ELEC|LIGHT|CHARGE", k, re.I)
               and k not in nombres])


# ---------------------------------------------------------------
# 3. Serie de LIGHTDENS por salida (para comparar con el LMA)
# ---------------------------------------------------------------
COLS = ["evento", "t", "suma", "suma_clip150", "maximo", "n_activos",
        "n_mayor_150", "enteros", "suma_light", "max_light",
        "n_light", "suma_lightdis"]


def _leer(ds, nombre):
    """Variable 2D (Time, y, x) como float64; NaN si no existe."""
    if nombre not in ds.variables:
        return None
    return np.asarray(np.ma.filled(ds.variables[nombre][0], 0.0),
                      dtype=np.float64)


def _fila_lightdens(arg):
    ev, f = arg
    with Dataset(f) as ds:
        y = _leer(ds, "LIGHTDENS")
        lg = _leer(ds, "LIGHT")        # orígenes de descarga
        ld = _leer(ds, "LIGHTDIS")     # fuentes de descarga
    act = y > 0
    nan = float("nan")
    return (ev, t_nombre(f), y.sum(), np.minimum(y, 150).sum(),
            y.max(), int(act.sum()), int((y > 150).sum()),
            bool(np.all(y[act] == np.round(y[act]))),
            nan if lg is None else lg.sum(),
            nan if lg is None else lg.max(),
            -1 if lg is None else int((lg > 0).sum()),
            nan if ld is None else ld.sum())


def serie_lightdens(eventos, ruta_csv="serie_electrica.csv",
                    solo=None, n_proc=N_PROC):
    """Una fila por salida con estadísticos de LIGHTDENS en todo el
    dominio (299x299, sin recortar). Guarda el CSV evento por evento
    y, si ya existe, saltea los eventos terminados. `solo` es una
    lista de claves de `eventos` para procesar solo esas."""
    previo = (pd.read_csv(ruta_csv, parse_dates=["t"])
              if os.path.exists(ruta_csv) else pd.DataFrame(columns=COLS))
    hechos = set(previo.evento.unique())
    pend = [ev for ev in sorted(solo or eventos) if ev not in hechos]
    pool = mp.get_context("fork").Pool(n_proc) if n_proc > 1 else None
    try:
        for ev in pend:
            filas = _mapear(_fila_lightdens,
                            [(ev, f) for f in eventos[ev]],
                            n_proc, f"Evento {ev}", pool)
            nuevo = pd.DataFrame(filas, columns=COLS)
            nuevo.to_csv(ruta_csv, mode="a", index=False,
                         header=not os.path.exists(ruta_csv))
    finally:
        if pool is not None:
            pool.close()
            pool.join()
    df = pd.read_csv(ruta_csv, parse_dates=["t"])
    act = df[df.n_activos > 0]
    print(f"\n{len(df)} salidas, {len(act)} con actividad "
          f"({len(act) / len(df):.1%}).")
    print("Valores de LIGHTDENS enteros en todas las salidas "
          f"activas: {bool(act.enteros.all())}")
    print("Máximo en un píxel:", df.maximo.max(),
          "| máx. de la suma por salida:", round(df.suma.max(), 1))
    por_ev = (df.groupby("evento")
                .agg(suma_max=("suma", "max"),
                     suma_mediana_activas=(
                         "suma", lambda s: s[s > 0].median()),
                     pix_max=("maximo", "max"),
                     pix_activos=("n_activos", "sum"),
                     pix_mayor_150=("n_mayor_150", "sum")))
    print(por_ev.round(1).to_string())
    print("\nTotal de píxeles activos:", int(df.n_activos.sum()),
          "| con más de 150:", int(df.n_mayor_150.sum()),
          f"({df.n_mayor_150.sum() / df.n_activos.sum():.2%})")
    for dia in ("2018-12-14", "2019-02-09"):      # días con LMA
        d = df[df.t.dt.strftime("%Y-%m-%d") == dia]
        if len(d):
            print(f"\n{dia}: suma de LIGHTDENS por salida (todo el "
                  f"dominio) máx {d.suma.max():.0f}, p95 "
                  f"{d.suma.quantile(.95):.0f}; recortada a 150: "
                  f"máx {d.suma_clip150.max():.0f}")
            print(f"{dia}: suma de LIGHT (orígenes) por salida: "
                  f"máx {d.suma_light.max():.0f}, p95 "
                  f"{d.suma_light.quantile(.95):.0f}; "
                  f"suma de LIGHTDIS: máx "
                  f"{d.suma_lightdis.max():.0f}")
    return df


# ---------------------------------------------------------------
# 4. Fracción de píxeles activos que supera 150 (en el dataset)
# ---------------------------------------------------------------
def _w_activos(f):
    with np.load(f) as z:
        if "y" not in z.files:           # p. ej. stats.npz
            return None
        y = np.squeeze(z["y"])
    return y[y > 0].astype(np.float32)


def fraccion_sobre_150(dir_dataset="../dataset_temporal", umbral=150,
                       n_proc=N_PROC):
    """Lee solo `y` (valores crudos) de cada muestra del dataset."""
    archivos = sorted(glob.glob(os.path.join(dir_dataset, "**",
                                             "*.npz"),
                                recursive=True))
    if not archivos:
        raise FileNotFoundError(
            f"No hay .npz en {os.path.abspath(dir_dataset)}. "
            f"Contenido: {sorted(os.listdir(dir_dataset))[:10]}")
    activos = [a for a in _mapear(_w_activos, archivos, n_proc,
                                  "Muestras") if a is not None]
    con_sobre = sum(bool((a > umbral).any()) for a in activos)
    v = np.concatenate(activos)
    out = {
        "muestras": len(activos),
        "pixeles_activos": int(v.size),
        "pixeles_sobre_umbral": int((v > umbral).sum()),
        "fraccion_pixeles": float((v > umbral).mean()),
        "muestras_con_algun_pixel_sobre_umbral": int(con_sobre),
        "percentiles_activos": {
            str(p): float(np.percentile(v, p))
            for p in (50, 90, 99, 99.9)},
        "maximo": float(v.max()),
    }
    print(json.dumps(out, indent=1, ensure_ascii=False))
    return out

In [ ]:
eventos = armar_eventos()   # debe decir: 16577 archivos, 17 eventos, paso 0:02:00

In [ ]:
serie = serie_lightdens(eventos)

In [ ]:
fraccion_sobre_150('../dataset_temporal')

In [ ]:
estructura_simulaciones(eventos)

In [ ]:
from wrf import getvar
from wrf.routines import _VALID_KARGS
from netCDF4 import Dataset
import numpy as np

_VALID_KARGS["maxdbz"] = ["use_varint", "use_liqskin"]   # el mismo parche de la celda 18

ds = Dataset(eventos['2018-12-14'][300])
m = getvar(ds, "mdbz", use_varint=True, use_liqskin=True).values
print("mín:", m.min(), "| máx:", m.max())
print("fracción < 0:", (m < 0).mean())
print("fracción == -30:", np.isclose(m, -30).mean())

In [ ]:
r = (serie.suma / serie.suma_light)[serie.suma_light > 0]
print(r.describe())

In [ ]:
eventos = armar_eventos()
r = verificar_recorte(eventos)

In [ ]:
from netCDF4 import Dataset
from wrf import getvar, to_np

ds = Dataset("/serafin/gcamilo/Trabajo_Final/salidasWRF2019/wrfout_d01_2019-03-04_12:28:00")
q = to_np(getvar(ds, "QCLOUD")); r = to_np(getvar(ds, "QRAIN"))
c = (slice(1, -1), slice(1, -1))          # mismo recorte (CROP=1) que el conjunto
print("max + max :", (q.max(0) + r.max(0))[c].max())
print("suma vert.:", (q + r).sum(0)[c].max())

In [ ]:
# Celda para Datasets.ipynb: histograma de LIGHTDENS en el conjunto de
# entrenamiento (valores ya restringidos en 150), para la figura de la
# Sección "Rangos y Normalización". Guarda figuras/datos/hist_lightdens.npz.
import glob, os
import numpy as np
from tqdm.auto import tqdm

archivos = sorted(glob.glob(f"{OUTPUT_DIR}/train/*.npz"))   # dataset final
EDGES_Y = np.arange(0, 151, 1.0)                  # (a) y en (0, 150]
EDGES_L = np.linspace(-2.31, 5.02, 75)            # (b) ln(y + 0,1)

counts_y = np.zeros(len(EDGES_Y) - 1, dtype=np.int64)
counts_l = np.zeros(len(EDGES_L) - 1, dtype=np.int64)
n_cero = n_total = 0
for f in tqdm(archivos):
    y = np.load(f)["y"].astype(np.float64)
    act = y[y > 0]
    n_total += y.size
    n_cero += y.size - act.size
    counts_y += np.histogram(act, bins=EDGES_Y)[0]
    counts_l += np.histogram(np.log(act + 0.1), bins=EDGES_L)[0]

os.makedirs("figuras/datos", exist_ok=True)
np.savez("figuras/datos/hist_lightdens.npz",
         edges_y=EDGES_Y, counts_y=counts_y,
         edges_l=EDGES_L, counts_l=counts_l,
         n_cero=n_cero, n_total=n_total)
print("columnas con actividad:", n_total - n_cero, "de", n_total)
print("máx y:", EDGES_Y[-1], "| activas por intervalo (a):", counts_y[:10])
print("fracción de activas con y < 5:",
      counts_y[:5].sum() / counts_y.sum(),
      "| y < 10:", counts_y[:10].sum() / counts_y.sum())

In [ ]:
# -*- coding: utf-8 -*-
"""
Opciones eléctricas del namelist, leídas de los atributos globales
de un wrfout, y descripción de LIGHTDENS/LIGHT guardada por el
modelo. Pegar en una celda de Datasets.ipynb (donde ya existe
`eventos`) y pasarme lo que imprime.
"""
import numpy as np
from netCDF4 import Dataset

ruta = eventos[sorted(eventos)[0]][0]
with Dataset(ruta) as ds:
    print("== Atributos globales (NSSL / ELEC / LIGHT) ==")
    for k in sorted(ds.ncattrs()):
        if any(s in k.upper() for s in ("NSSL", "ELEC", "LIGHT",
                                        "LTNG", "DISCH", "BRK")):
            print(f"{k:28s} {np.asarray(getattr(ds, k)).tolist()}")
    print("\n== Variables eléctricas ==")
    for v in ("LIGHTDENS", "LIGHT", "LIGHTDIS", "INDUC", "NONINDUC"):
        if v in ds.variables:
            x = ds.variables[v]
            print(v, "|", getattr(x, "description", ""), "|",
                  getattr(x, "units", ""))

In [ ]:
# -*- coding: utf-8 -*-
"""
Histograma de LIGHT y de LIGHTDENS (sin restringir) de los wrfout, para
ver si LIGHT presenta picos en valores periódicos como LIGHTDENS.

Pegar en una celda de Datasets.ipynb, donde ya existe `eventos`
(celda "Corridas continuas"). Usa un subconjunto de las salidas
(una de cada PASO) para que no demore. Guarda
figuras/datos/hist_light_lightdens.npz y imprime un resumen.
"""
import os
import numpy as np
from netCDF4 import Dataset
from tqdm.auto import tqdm

PASO = 3                        # usar una de cada PASO salidas
VMAX = 1200
archivos = sorted({f for fs in eventos.values() for f in fs})[::PASO]

cuentas = {"LIGHT": np.zeros(VMAX + 2, np.int64),
           "LIGHTDENS": np.zeros(VMAX + 2, np.int64)}
no_enteros = {"LIGHT": 0, "LIGHTDENS": 0}
for f in tqdm(archivos):
    with Dataset(f) as ds:
        for v in cuentas:
            x = np.squeeze(np.asarray(ds.variables[v][:], np.float64))
            x = x[x > 0]
            no_enteros[v] += int((x != np.round(x)).sum())
            r = np.minimum(np.round(x).astype(np.int64), VMAX + 1)
            cuentas[v] += np.bincount(r, minlength=VMAX + 2)

os.makedirs("figuras/datos", exist_ok=True)
np.savez("figuras/datos/hist_light_lightdens.npz",
         light=cuentas["LIGHT"], lightdens=cuentas["LIGHTDENS"],
         paso=PASO, n_archivos=len(archivos))
print(len(archivos), "salidas usadas")
for v, c in cuentas.items():
    tot = c[1:].sum()
    print(f"\n== {v} ==  columnas con actividad: {tot}  "
          f"(valores no enteros: {no_enteros[v]})")
    print("máximo:", int(np.nonzero(c)[0].max()))
    top = np.argsort(c[1:])[::-1][:12] + 1
    print("12 valores más frecuentes:",
          [(int(t), int(c[t])) for t in top])
    m20 = c[20::20].sum()
    print(f"fracción en múltiplos de 20: {m20 / tot:.3f} "
          f"(esperada sin picos, aprox. {len(c[20::20]) / VMAX:.3f} "
          "del rango)")
    for k in (20, 40, 60, 80):
        print(f"  valor {k}: {c[k]} | vecinos {c[k-1]} y {c[k+1]}")

In [ ]:
# -*- coding: utf-8 -*-
"""Grafica la distribución de LIGHT y LIGHTDENS a partir de
datos/hist_light_lightdens.npz (lo genera celda_hist_light.py).
Uso: python graficar_hist_light.py  ->  hist_light_lightdens.png"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

d = np.load("figuras/datos/hist_light_lightdens.npz")
vars_ = [("LIGHT", d["light"]), ("LIGHTDENS", d["lightdens"])]

fig, ax = plt.subplots(2, 2, figsize=(11, 6.5))
for j, (nombre, c) in enumerate(vars_):
    x = np.arange(len(c))
    # fila 1: valores 1 a 100, escala log en y, picos marcados
    a = ax[0, j]
    m = (x >= 1) & (x <= 100)
    a.bar(x[m], c[m], width=0.9, color="#4c72b0")
    for k in (20, 40, 60, 80, 100):
        a.axvline(k, color="#c44e52", ls=":", lw=0.8)
    a.set_yscale("log")
    a.set_title(f"{nombre}: valores de 1 a 100")
    a.set_xlabel("descargas por columna")
    a.set_ylabel("columnas (escala log)")
    # fila 2: rango completo, log-log
    b = ax[1, j]
    m = (x >= 1) & (c > 0)
    b.plot(x[m], c[m], ".", ms=3, color="#4c72b0")
    b.set_xscale("log"); b.set_yscale("log")
    b.set_title(f"{nombre}: rango completo (hasta 1.200)")
    b.set_xlabel("descargas por columna (escala log)")
    b.set_ylabel("columnas (escala log)")
fig.suptitle("Distribución de LIGHT y LIGHTDENS en los wrfout "
             f"(1 de cada {int(d['paso'])} salidas, {int(d['n_archivos'])} archivos)")
fig.tight_layout()
fig.savefig("figuras/hist_light_lightdens.png", dpi=150)
print("guardado hist_light_lightdens.png")